# 프라이빗 RAG 구현하기 [프로젝트] — 입학사정관 규정 챗봇


## Step 0 : 설치와 준비

필요한 라이브러리를 설치합니다. 강의안과 같이 버전을 지정해서, 몇 달 뒤에 최신 버전이 설치되어 코드가 깨지는 일을 막습니다. 이 노트북은 GPU 런타임에서 실행하며, API 키 등록 단계는 없습니다.

In [ ]:
# 라이브러리 버전을 지정해 설치합니다
# 버전을 지정하지 않으면 몇 달 뒤에는 최신 버전이 설치되고, 함수 이름이나 인자가 달라져 이 코드가 동작하지 않을 수 있습니다
!pip install -q "langchain==0.3.*" "langchain-community==0.3.*" "langchain-text-splitters==0.3.*" \
                "langchain-experimental==0.3.*" "langchain-huggingface==0.3.*"
!pip install -q "sentence-transformers==5.7.*" "qdrant-client==1.12.*"
!pip install -q "transformers==4.57.*" "accelerate==1.14.*" "bitsandbytes==0.50.*"
# [변경] .docx 파일을 읽기 위한 라이브러리를 추가합니다
!pip install -q "python-docx==1.*"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is inc

설치 중 빨간 글씨로 pip 의존성 경고(`pip's dependency resolver …`)가 나올 수 있습니다. 경고는 오류가 아니므로 다음 셀이 정상적으로 실행되면 그대로 진행하면 됩니다. 뒤에서 모델을 처음 내려받을 때 나오는 `HF_TOKEN` 경고도 마찬가지입니다(공개 모델은 토큰 없이 받을 수 있습니다).

In [ ]:
import torch
import transformers, sentence_transformers, qdrant_client

# GPU 확인. T4, VRAM 약 15~16GB로 표시되면 정상입니다.
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 설치된 버전을 기록해둡니다. 문제가 생기면 버전부터 확인해 보세요.
print(f"transformers {transformers.__version__} / sentence-transformers {sentence_transformers.__version__}")

GPU : Tesla T4
VRAM : 15.6 GB
transformers 4.57.6 / sentence-transformers 5.7.0


In [ ]:
# 이후 여러 단계에서 공통으로 쓰는 라이브러리를 한 번에 가져옵니다
import os          # 파일 경로와 환경변수를 다루는 도구
import re          # 정규표현식(글자 패턴 찾기) 도구
import time        # 걸린 시간을 재는 도구
import numpy as np # 숫자 배열·행렬 계산 도구
import pandas as pd            # 표(DataFrame)를 만들고 보여주는 도구
from IPython.display import display   # 표를 예쁘게 출력하는 함수

## Step 1 : 문서 준비

이번 프로젝트의 문서는 「홍익대학교 입학사정관 임용 및 운영에 관한 규정」 `.docx` 파일입니다. 장(章) → 조(條) → 항(項, ①②③) → 호(號, 1. 2. 3.)로 이어지는 **계층 구조가 뚜렷한 규정 문서**라서, 강의안의 사내 규정과 같은 성격입니다.

다만 강의안처럼 마크다운으로 정리된 문서가 아니라서, 먼저 파일을 열어 구조를 확인하고 청킹에 쓸 수 있는 형태로 바꿔야 합니다. 원본을 열어 보니 아래 네 가지를 처리해야 했습니다.

| # | 문제 | 예시 | 처리 방법 |
| --- | --- | --- | --- |
| 1 | "조"가 제목 스타일이 아니라 일반 본문 | `제1조(목적) 본 규정은 …` | `### 제N조(제목)` 헤더를 직접 만들어 넣음 |
| 2 | 호(1. 2. 3.)의 번호가 워드의 자동 번호라 글자로 읽으면 사라짐 | "전형(또는 평가)위원 구성"만 읽힘 | 연속된 목록 문단에 번호를 다시 붙임 |
| 3 | 문장 중간에서 끊긴 문단 | `…갖추어야 한` / `다.` , `<개정` / `2011.3.1>` | 앞 문단에 이어 붙임 |
| 4 | 단어 중간에 끼어든 공백과 오타 | `평가하거 나`, `교 수입학사정관`, 제17조 ⑦이 `➀`로 적힘 | 확인한 목록만 골라서 고침 |

4번은 검색 품질에 직접 영향을 줍니다. "평가하거 나"처럼 잘린 단어는 임베딩 모델이 다른 토큰으로 쪼개기 때문입니다. 대신 원문을 함부로 바꾸지 않으려고, **자동 교정이 아니라 눈으로 확인한 목록**만 고치고 고친 횟수를 표로 남깁니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# [변경] 강의안은 %%writefile로 문서를 직접 만들었지만, 이번에는 직접 고른 .docx 파일을 불러옵니다
DOCX_PATH = '/content/drive/MyDrive/aiffel_data/hongik_regulation.docx'   # 드라이브에 올린 경로 (본인 경로에 맞게 바꾸세요)

if not os.path.exists(DOCX_PATH):
    # 드라이브에 파일이 없으면 컴퓨터에서 직접 올릴 수 있는 창을 띄웁니다
    from google.colab import files
    uploaded = files.upload()
    DOCX_PATH = list(uploaded.keys())[0]     # 올린 파일의 이름을 경로로 사용합니다

print("문서 경로 :", DOCX_PATH)

Mounted at /content/drive
문서 경로 : /content/drive/MyDrive/aiffel_data/hongik_regulation.docx


In [ ]:
import docx                          # python-docx : .docx 파일을 읽는 라이브러리
from collections import Counter      # 종류별 개수를 세는 도구

word_doc = docx.Document(DOCX_PATH)  # 워드 파일을 엽니다
paragraphs = word_doc.paragraphs     # 문서 전체를 "문단(paragraph)" 목록으로 가져옵니다

print(f"문단 수 : {len(paragraphs)}")
# 문단마다 붙어 있는 스타일(제목/본문/목록)의 종류별 개수를 셉니다
print(Counter(p.style.name for p in paragraphs))

# 앞쪽 문단 8개를 스타일과 함께 확인합니다. "제1조"가 Heading이 아니라 Body Text인 것을 확인해 보세요
for i, p in enumerate(paragraphs[:8]):
    print(i, f"[{p.style.name}]", repr(p.text[:40]))

문단 수 : 146
Counter({'Body Text': 104, 'List Paragraph': 29, 'Heading 1': 7, 'Normal': 5, 'Title': 1})
0 [Title] '홍익대학교 입학사정관 임용 및 운영에 관한 규정'
1 [Body Text] ''
2 [Heading 1] '제1장 총칙'
3 [Body Text] ''
4 [Body Text] '제1조(목적) 본 규정은 홍익대학교(이하 ‘본교’라 한다) 신입생선발 과'
5 [Body Text] ''
6 [Body Text] '제2조(입학사정관) ① 교육과학기술부의 입학사정관제 지원사업(기간제 및 '
7 [Body Text] '② 입학사정관 업무를 전담하기 위하여 본교에서 상근 직원으로 임용한 경우'


출력에서 확인할 것은 두 가지입니다. `Heading 1` 스타일은 "제1장 총칙" 같은 **장**뿐이고, "제1조(목적) …"는 `Body Text`(일반 본문)입니다. 그래서 강의안처럼 `###` 헤더가 이미 있는 문서가 아닙니다. 아래 코드가 이 부분을 채워 넣습니다.

변환은 두 단계입니다. ① 문단을 하나씩 읽으면서 "제목/조/본문/목록"으로 종류를 붙이고(`docx_to_blocks`), ② 그 결과를 마크다운 글로 이어 붙입니다(`blocks_to_markdown`).

In [ ]:
# ---------------------------------------------------------------
# [변경] 강의안에 없는 새 코드 : .docx → 마크다운 변환
# ---------------------------------------------------------------

# 글자 패턴(정규표현식) 정의
MARKER_RE  = re.compile(r"^[①②③④⑤⑥⑦⑧⑨➀]")                 # 문단이 항 번호(①②③…)로 시작하는가
ARTICLE_RE = re.compile(r"^제(\d+)조\s*\(([^)]*)\)\s*(.*)$")    # "제N조(제목) 나머지 본문" 형태인가
CHAPTER_RE = re.compile(r"^제(\d+)장\s+.+$")                    # "제N장 제목" 형태인가
NOTE_RE    = re.compile(r"^\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\.?>")  # "2011.3.1>" 처럼 개정 표시의 꼬리인가

# 눈으로 확인한 원문 오류 목록 (원문, 고친 글자)
CLEAN_RULES = [
    ("평가하거 나", "평가하거나"),
    ("정 하기", "정하기"),
    ("등 에 관한", "등에 관한"),
    ("교 수입학사정관", "교수입학사정관"),
    ("위촉입 학사정관", "위촉입학사정관"),
    ("필요 한 사항", "필요한 사항"),
    ("지원자 를", "지원자를"),
    ("교육과 학기술부", "교육과학기술부"),
    ("제청으 로", "제청으로"),
    ("입학사정관 제 지원사업", "입학사정관제 지원사업"),
    ("인사위 원회", "인사위원회"),
    ("구 성한다", "구성한다"),
    ("대행토 록", "대행토록"),
    ("승계규 정", "승계규정"),
    ("개 진하도록", "개진하도록"),
    ("명 칭 여하", "명칭 여하"),
    ("교원인사위원 회의", "교원인사위원회의"),
    ("것으 로 한다", "것으로 한다"),
    ("책임입학사정관 을", "책임입학사정관을"),
    ("초 과할", "초과할"),
    ("➀", "⑦"),                       # 제17조의 일곱 번째 항 번호가 다른 글자(➀)로 적혀 있음
]


def normalize_ws(text):
    """줄바꿈·탭·여러 칸 공백을 공백 한 칸으로 줄입니다"""
    return re.sub(r"\s+", " ", text.replace(" ", " ")).strip()


def docx_to_blocks(paragraphs):
    """문단 목록을 (종류, 글) 목록으로 바꿉니다. 종류 : title / chapter / appendix / article / body / list"""
    blocks = []
    appendix_no = 0                                  # 부칙이 몇 번째인지 셉니다
    for p in paragraphs:
        text = normalize_ws(p.text)                  # 공백을 정리한 문단 글
        if not text:                                 # 빈 문단은 건너뜁니다
            continue
        style = p.style.name                         # 문단 스타일 이름
        compact = text.replace(" ", "")              # 공백을 모두 뺀 글 ("부  칙" → "부칙" 비교용)
        last_kind = blocks[-1][0] if blocks else None   # 바로 앞에 만든 블록의 종류

        if style == "Title":                         # 문서 제목
            blocks.append(("title", text))
            continue

        if compact.startswith("부칙"):               # 부칙은 같은 제목이 반복되므로 번호를 붙여 구분합니다
            appendix_no += 1
            blocks.append(("appendix", f"부칙 {appendix_no}"))
            rest = re.sub(r"^부\s*칙\s*", "", text)   # "부 칙 (시행일) …"처럼 제목 뒤에 본문이 붙은 경우
            if rest:
                blocks.append(("body", rest))
            continue

        if style == "Heading 1" or CHAPTER_RE.match(text):   # 장
            blocks.append(("chapter", text))
            continue

        m = ARTICLE_RE.match(text)                   # 조 : "제1조(목적) 본 규정은 …"
        if m:
            blocks.append(("article", f"제{m.group(1)}조({m.group(2)})"))
            if m.group(3):                           # 제목 뒤에 이어진 본문이 있으면 따로 담습니다
                blocks.append(("body", m.group(3)))
            continue

        if style == "List Paragraph":                # 호(1. 2. 3.) : 워드의 자동 번호라서 번호를 다시 붙입니다
            n = 1
            if last_kind == "list":                  # 바로 앞도 목록이면 앞 번호 + 1
                n = int(re.match(r"^(\d+)\.", blocks[-1][1]).group(1)) + 1
            blocks.append(("list", f"{n}. {text}"))
            continue

        # 항 번호(①)로 시작하지 않는 일반 문단은 앞 문단에서 끊긴 문장의 꼬리입니다 → 앞 문단에 이어 붙입니다
        if last_kind == "body" and not MARKER_RE.match(text):
            prev = blocks[-1][1]
            if NOTE_RE.match(text):
                joiner = " "                         # "<개정" + " " + "2011.3.1>"
            elif re.search(r"[가-힣]$", prev):
                joiner = ""                          # "…갖추어야 한" + "다." (한글 단어가 끊긴 경우는 붙여 씁니다)
            else:
                joiner = " "                         # "…다만," + " " + "「교육공무원법」…"
            blocks[-1] = ("body", prev + joiner + text)
            continue

        blocks.append(("body", text))                # 나머지는 일반 본문
    return blocks


def blocks_to_markdown(blocks):
    """블록 목록을 마크다운 글로 이어 붙입니다 (# 문서, ## 장, ### 조/부칙)"""
    lines = []
    appendix_started = False
    for kind, text in blocks:
        if kind == "title":
            lines.append(f"# {text}")
        elif kind == "chapter":
            lines.append(f"## {text}")
        elif kind == "appendix":
            if not appendix_started:                 # 부칙 전체를 하나의 "장"처럼 묶습니다
                lines.append("## 부칙")
                appendix_started = True
            lines.append(f"### {text}")
        elif kind == "article":
            lines.append(f"### {text}")
        else:                                        # body, list
            lines.append(text)
    return "\n\n".join(lines)

print("변환 함수 준비 완료")

변환 함수 준비 완료


In [ ]:
blocks = docx_to_blocks(paragraphs)                 # ① 문단 → 블록
markdown_text = blocks_to_markdown(blocks)          # ② 블록 → 마크다운 글

# ③ 원문 오류 정제 : 몇 번 고쳤는지 표로 남깁니다
rows = []
for old, new in CLEAN_RULES:
    rows.append((old, new, markdown_text.count(old)))   # 고치기 전에 몇 번 나오는지 셉니다
    markdown_text = markdown_text.replace(old, new)     # 실제로 고칩니다
markdown_text = re.sub(r"([․·])\s+", r"\1", markdown_text)              # "수집․정리․ 분석" → 가운뎃점 뒤의 불필요한 공백 제거
markdown_text = re.sub(r"([①②③④⑤⑥⑦⑧⑨])(?=\S)", r"\1 ", markdown_text)  # "①입학" → "① 입학" (항 번호 뒤에 공백 통일)

display(pd.DataFrame(rows, columns=["원문", "고친 글자", "고친 횟수"]))

# 강의안과 같은 방식으로 파일에 저장했다가 다시 읽어서 document 로 씁니다
with open("hongik_regulation.md", "w", encoding="utf-8") as f:
    f.write(markdown_text)

with open("hongik_regulation.md", encoding="utf-8") as f:
    document = f.read()

print(f"문서 길이 : {len(document)}자")

,원문,고친 글자,고친 횟수
0,평가하거 나,평가하거나,1
1,정 하기,정하기,1
2,등 에 관한,등에 관한,1
3,교 수입학사정관,교수입학사정관,1
4,위촉입 학사정관,위촉입학사정관,1
5,필요 한 사항,필요한 사항,1
6,지원자 를,지원자를,1
7,교육과 학기술부,교육과학기술부,1
8,제청으 로,제청으로,1
9,입학사정관 제 지원사업,입학사정관제 지원사업,1


문서 길이 : 5705자


In [ ]:
# 변환 결과 점검 : 조·장·부칙의 개수가 원문과 맞는지 확인합니다
n_article  = len(re.findall(r"^### 제\d+조\(", document, flags=re.M))
n_chapter  = len(re.findall(r"^## 제\d+장", document, flags=re.M))
n_appendix = len(re.findall(r"^### 부칙 \d+", document, flags=re.M))
print(f"조 {n_article}개 / 장 {n_chapter}개 / 부칙 {n_appendix}개   (원문 기준 예상 : 21 / 5 / 4)")
assert (n_article, n_chapter, n_appendix) == (21, 5, 4), "조·장·부칙 개수가 예상과 다릅니다. 변환 규칙을 다시 확인하세요"

# 정제 뒤에도 남은 "단어 뒤에 홀로 떨어진 조사"가 있는지 찾아봅니다 (아무것도 안 나오면 정상)
print("남은 의심 구간 :", re.findall(r"[가-힣]+ [을를이가은는에로과와의](?= |,|\.|$)", document))

# 변환된 문서의 앞부분을 확인합니다
print()
print(document[:900])

조 21개 / 장 5개 / 부칙 4개   (원문 기준 예상 : 21 / 5 / 4)
남은 의심 구간 : []

# 홍익대학교 입학사정관 임용 및 운영에 관한 규정

## 제1장 총칙

### 제1조(목적)

본 규정은 홍익대학교(이하 ‘본교’라 한다) 신입생선발 과정에서 지원자를 평가하거나 입학시험 등 전형과 관련된 업무에 참여할 입학사정관의 임용, 위촉 및 운영에 관한 사항을 정하기 위함을 목적으로 한다.

### 제2조(입학사정관)

① 교육과학기술부의 입학사정관제 지원사업(기간제 및 단시간근로자 보호 등에 관한 법률 제4조 1항 1호의 사업을 의미한다)을 수행하기 위하여, 입학사정관으로 상근인 전임 입학사정관과 교수입학사정관, 비상근인 위촉입학사정관과 초빙입학사정관을 둔다.

② 입학사정관 업무를 전담하기 위하여 본교에서 상근 직원으로 임용한 경우, 전임입학사정관으로 한다.

③ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 상근 보직으로 임명한 경우, 교수입학사정관으로 한다.

④ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 위촉입학사정관으로 한다.

⑤ 본교의 교원이 아니지만 본교의 요청에 의하여 한시적으로 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 초빙입학사정관으로 한다.

### 제3조(사정관제 전형)

① 입학사정관제 전형을 통하여 신입생을 선발할 모집단위, 모집인원, 모집 시기, 전형요소, 전형방법 등의 세부사항은 입학관리본부장(이하 ‘본부장’이라 한다)이 총장의 승인을 받아 따로 정하고, 주요사항은 모집요강을 통하여 사전에 공고한다.

② 입학사정관제 전형은 모집요강과 사정원칙에 입각하여 시행한다.

### 제4조(사정원칙)

① 입학사정관제 전형의 시행과 관련한 세부원칙과 방법은 사정원칙으로 정한다.

② 사정원칙은 다음 각 호에 대하여 규정한다.

1. 전형(또는 평가)위원 구성

2. 전형요소 및 전형방법

3


제10조, 제11조처럼 조마다 항(①②③)과 호(1. 2.)가 이어지는 구조가 마크다운 글로 잘 옮겨졌는지 위 출력에서 확인해 주세요.

강의안의 문서(`company_policy.md`)와 같은 모양이 되었으므로, 다음 Step부터는 강의안 코드를 거의 그대로 쓸 수 있습니다.

## Step 2 : 청킹 전략 비교

같은 문서에 네 가지 청킹 전략을 적용하고, 결과가 어떻게 다른지 비교합니다. 강의안과 같은 순서(고정 크기 → 재귀적 → 문서 구조 기반 → 시맨틱)로 진행하고, 마지막에 이 문서에 어떤 전략이 맞는지 **숫자로** 비교합니다.

#### 고정 크기 청킹

글자 수 기준으로 기계적으로 자릅니다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",        # 구분자 없이 글자 수로만 자릅니다
    chunk_size=300,
    chunk_overlap=50,
)

fixed_chunks = fixed_splitter.split_text(document)
print(f"청크 수 : {len(fixed_chunks)}")

청크 수 : 23


In [ ]:
# 앞쪽 청크 3개를 확인합니다
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"[청크 {i}] ({len(chunk)}자)")
    print(chunk)
    print("-" * 50)

[청크 0] (300자)
# 홍익대학교 입학사정관 임용 및 운영에 관한 규정

## 제1장 총칙

### 제1조(목적)

본 규정은 홍익대학교(이하 ‘본교’라 한다) 신입생선발 과정에서 지원자를 평가하거나 입학시험 등 전형과 관련된 업무에 참여할 입학사정관의 임용, 위촉 및 운영에 관한 사항을 정하기 위함을 목적으로 한다.

### 제2조(입학사정관)

① 교육과학기술부의 입학사정관제 지원사업(기간제 및 단시간근로자 보호 등에 관한 법률 제4조 1항 1호의 사업을 의미한다)을 수행하기 위하여, 입학사정관으로 상근인 전임 입학사정관과 교수입학사정관, 비상근인
--------------------------------------------------
[청크 1] (300자)
한다)을 수행하기 위하여, 입학사정관으로 상근인 전임 입학사정관과 교수입학사정관, 비상근인 위촉입학사정관과 초빙입학사정관을 둔다.

② 입학사정관 업무를 전담하기 위하여 본교에서 상근 직원으로 임용한 경우, 전임입학사정관으로 한다.

③ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 상근 보직으로 임명한 경우, 교수입학사정관으로 한다.

④ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 위촉입학사정관으로 한다.

⑤ 본교의 교원이 아니지만 본교의 요청에 의하여 한시적으로 입학사정관의 직
--------------------------------------------------
[청크 2] (300자)
관으로 한다.

⑤ 본교의 교원이 아니지만 본교의 요청에 의하여 한시적으로 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 초빙입학사정관으로 한다.

### 제3조(사정관제 전형)

① 입학사정관제 전형을 통하여 신입생을 선발할 모집단위, 모집인원, 모집 시기, 전형요소, 전형방법 등의 세부사항은 입학관리본부장(이하 ‘본부장’이라 한다)이 총장의 승인을 받아 따로 정하고, 주요사항은 모집요강을 통하여 사전에 공고한다.

② 입학사정관제 전형은 

#### 재귀적 청킹

구분자 우선순위(문단 → 줄 → 문장 → 단어 → 글자)에 따라 가능한 한 의미 단위를 보존하며 자릅니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
)

recursive_chunks = recursive_splitter.split_text(document)
print(f"청크 수 : {len(recursive_chunks)}")

청크 수 : 23


In [ ]:
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"[청크 {i}] ({len(chunk)}자)")
    print(chunk)
    print("-" * 50)

[청크 0] (183자)
# 홍익대학교 입학사정관 임용 및 운영에 관한 규정

## 제1장 총칙

### 제1조(목적)

본 규정은 홍익대학교(이하 ‘본교’라 한다) 신입생선발 과정에서 지원자를 평가하거나 입학시험 등 전형과 관련된 업무에 참여할 입학사정관의 임용, 위촉 및 운영에 관한 사항을 정하기 위함을 목적으로 한다.

### 제2조(입학사정관)
--------------------------------------------------
[청크 1] (275자)
### 제2조(입학사정관)

① 교육과학기술부의 입학사정관제 지원사업(기간제 및 단시간근로자 보호 등에 관한 법률 제4조 1항 1호의 사업을 의미한다)을 수행하기 위하여, 입학사정관으로 상근인 전임 입학사정관과 교수입학사정관, 비상근인 위촉입학사정관과 초빙입학사정관을 둔다.

② 입학사정관 업무를 전담하기 위하여 본교에서 상근 직원으로 임용한 경우, 전임입학사정관으로 한다.

③ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 상근 보직으로 임명한 경우, 교수입학사정관으로 한다.
--------------------------------------------------
[청크 2] (293자)
④ 본교의 교원으로서 일정기간 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 위촉입학사정관으로 한다.

⑤ 본교의 교원이 아니지만 본교의 요청에 의하여 한시적으로 입학사정관의 직무수행을 위하여 비상근으로 위촉한 경우, 초빙입학사정관으로 한다.

### 제3조(사정관제 전형)

① 입학사정관제 전형을 통하여 신입생을 선발할 모집단위, 모집인원, 모집 시기, 전형요소, 전형방법 등의 세부사항은 입학관리본부장(이하 ‘본부장’이라 한다)이 총장의 승인을 받아 따로 정하고, 주요사항은 모집요강을 통하여 사전에 공고한다.
--------------------------------------------------


#### 문서 구조 기반 청킹

Step 1에서 만든 마크다운 헤더(`#` 문서 / `##` 장 / `###` 조)를 그대로 분할 기준으로 사용합니다. 강의안 코드와 완전히 같고, 달라진 것은 문서를 만드는 과정(Step 1)뿐입니다.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "문서"),
    ("##", "장"),
    ("###", "조항"),
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(document)

print(f"청크 수 : {len(md_chunks)}")

청크 수 : 25


In [ ]:
for chunk in md_chunks[:3]:
    print(f"[메타데이터] {chunk.metadata}")
    print(chunk.page_content[:120])
    print("-" * 50)

[메타데이터] {'문서': '홍익대학교 입학사정관 임용 및 운영에 관한 규정', '장': '제1장 총칙', '조항': '제1조(목적)'}
본 규정은 홍익대학교(이하 ‘본교’라 한다) 신입생선발 과정에서 지원자를 평가하거나 입학시험 등 전형과 관련된 업무에 참여할 입학사정관의 임용, 위촉 및 운영에 관한 사항을 정하기 위함을 목적으로 한다.
--------------------------------------------------
[메타데이터] {'문서': '홍익대학교 입학사정관 임용 및 운영에 관한 규정', '장': '제1장 총칙', '조항': '제2조(입학사정관)'}
① 교육과학기술부의 입학사정관제 지원사업(기간제 및 단시간근로자 보호 등에 관한 법률 제4조 1항 1호의 사업을 의미한다)을 수행하기 위하여, 입학사정관으로 상근인 전임 입학사정관과 교수입학사정관, 비상근인 위촉입학
--------------------------------------------------
[메타데이터] {'문서': '홍익대학교 입학사정관 임용 및 운영에 관한 규정', '장': '제1장 총칙', '조항': '제3조(사정관제 전형)'}
① 입학사정관제 전형을 통하여 신입생을 선발할 모집단위, 모집인원, 모집 시기, 전형요소, 전형방법 등의 세부사항은 입학관리본부장(이하 ‘본부장’이라 한다)이 총장의 승인을 받아 따로 정하고, 주요사항은 모집요강을 
--------------------------------------------------


청크가 **조 단위**로 잘렸고, 각 청크에 장·조 이름이 **메타데이터**로 붙었습니다. 이 정보는 뒤에서 출처 표기와 필터 검색에 쓰입니다. 조 21개와 부칙 4개, 합쳐서 25개의 청크가 나와야 합니다.

#### 잠깐, 글자 수가 아니라 토큰 수

임베딩 모델의 입력 상한은 **토큰 기준**입니다. 상한을 넘겨도 에러 없이 뒷부분이 조용히 잘리기 때문에(트랙 3의 "실패 1"), 청크마다 실제 토큰 수를 재 봅니다.

In [ ]:
from transformers import AutoTokenizer

# 두 모델의 토크나이저로 같은 청크의 토큰 수를 비교합니다
tok_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tok_bge = AutoTokenizer.from_pretrained("BAAI/bge-m3")

print(f"{'청크':>4} {'글자':>6} {'E5 토큰':>8} {'BGE 토큰':>9}  조항")
print("-" * 60)
max_e5, max_bge = 0, 0                     # [변경] 가장 긴 청크의 토큰 수를 기록합니다
for i, c in enumerate(md_chunks):
    text = c.page_content
    n_e5 = len(tok_e5.encode(text))
    n_bge = len(tok_bge.encode(text))
    max_e5, max_bge = max(max_e5, n_e5), max(max_bge, n_bge)
    flag = " <- E5 상한(512) 초과" if n_e5 > 512 else ""
    print(f"{i:>4} {len(text):>6} {n_e5:>8} {n_bge:>9}  {c.metadata.get('조항','')}{flag}")

print(f"\n가장 긴 청크의 토큰 수 : E5 {max_e5} / BGE {max_bge}  (E5 상한 512, BGE-M3 상한 8192)")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

  청크     글자    E5 토큰    BGE 토큰  조항
------------------------------------------------------------
   0    113       61        61  제1조(목적)
   1    405      248       248  제2조(입학사정관)
   2    171      100       100  제3조(사정관제 전형)
   3    198      100       100  제4조(사정원칙)
   4    268      154       154  제5조(입학사정관실)
   5    499      248       248  제6조(입학사정관의 업무)
   6    159      102       102  제7조(입학사정관의 수)
   7    203      122       122  제8조(입학사정관의 임용 및 위촉)
   8    230      134       134  제9조(자격 기준)
   9    220      136       136  제10조(구비서류)
  10    317      196       196  제11조(계약기간)
  11    257      144       144  제12조(재계약)
  12    189      110       110  제13조(근무성적 평가)
  13    271      144       144  제14조(인사위원회)
  14    209      118       118  제15조(구성 및 임기)
  15    182      110       110  제16조(위원장)
  16    394      227       227  제17조(위원회의 소집 및 운영)
  17    155       84        84  제18조(소위원회)
  18    206      116       116  제19조(입학사정관의 활동 및 교육)
  19     13        9         9  제20조(위임전결)
  20  

#### 시맨틱 청킹

의미가 바뀌는 지점에서 자릅니다. 프라이빗 환경이므로 임베딩 모델은 로컬 모델(가벼운 e5-small)을 도구로만 씁니다.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# 가벼운 다국어 임베딩 모델 (약 470MB)
embeddings_for_chunking = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cuda"},
)

semantic_splitter = SemanticChunker(
    embeddings_for_chunking,
    breakpoint_threshold_type="percentile",   # 유사도 하락 지점을 백분위 기준으로 판단
)

semantic_chunks = semantic_splitter.split_text(document)
print(f"청크 수 : {len(semantic_chunks)}")

# [변경] 청킹에만 쓴 모델이므로 GPU 메모리를 돌려줍니다 (뒤에서 LLM과 임베딩 모델을 함께 올려야 하기 때문)
del embeddings_for_chunking, semantic_splitter
torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

청크 수 : 6


#### 네 전략 나란히 비교

강의안의 비교표(청크 수, 평균·최소·최대 길이)에 이 문서에 맞는 **두 개의 열을 추가**했습니다.

- **조 보존율** : 원문의 조(條) 25개 중, 본문 전체가 하나의 청크 안에 온전히 들어 있는 조의 비율. 조 하나가 두 청크로 잘렸으면 보존되지 않은 것으로 셉니다.
- **조 혼합 청크** : 한 청크 안에 서로 다른 조의 제목(`### 제N조(`)이 둘 이상 들어 있는 청크 수. 서로 다른 주제가 한 벡터에 섞인다는 뜻입니다. (구조 기반은 조 제목이 메타데이터로 빠져 있어서 정의상 0입니다.)

In [ ]:
# [변경] 이 문서에 맞춘 평가 지표 두 개
def norm(s):
    return re.sub(r"\s+", "", s)                          # 공백·줄바꿈을 모두 지워서 글자만 비교합니다

article_bodies = [norm(c.page_content) for c in md_chunks]   # 원문 조·부칙 25개의 본문 (구조 기반 청크가 곧 정답지)

def intact_ratio(chunk_list):
    """조 본문이 하나의 청크 안에 통째로 들어 있는 비율"""
    normed = [norm(c) for c in chunk_list]
    ok = sum(any(body in n for n in normed) for body in article_bodies)
    return ok / len(article_bodies)

def mixed_count(chunk_list):
    """서로 다른 조 제목이 2개 이상 들어 있는 청크의 수"""
    return sum(len(re.findall(r"^###\s*제\d+조\(", c, flags=re.M)) >= 2 for c in chunk_list)


strategies = {
    "고정 크기": fixed_chunks,
    "재귀적": recursive_chunks,
    "구조 기반": [c.page_content for c in md_chunks],
    "시맨틱": semantic_chunks,
}

print(f"{'전략':<8} {'청크 수':>6} {'평균 길이':>8} {'최소':>6} {'최대':>6} {'조 보존율':>9} {'조 혼합 청크':>11}")
print("-" * 68)
for name, chunk_list in strategies.items():
    lengths = [len(c) for c in chunk_list]
    print(f"{name:<8} {len(chunk_list):>6} {sum(lengths)/len(lengths):>8.0f} {min(lengths):>6} {max(lengths):>6} "
          f"{intact_ratio(chunk_list):>9.0%} {mixed_count(chunk_list):>11}")

전략         청크 수    평균 길이     최소     최대     조 보존율     조 혼합 청크
--------------------------------------------------------------------
고정 크기        23      295    205    300       56%          10
재귀적          23      264    103    298       48%           5
구조 기반        25      208     13    499      100%           0
시맨틱           6      940    217   2171       80%           2


In [ ]:
# 정성 비교 : "재계약 신청은 언제까지?"의 근거인 제12조 2항("만료 45일 전까지")이
# 각 전략에서 어떤 모습의 청크로 존재하는지 확인합니다

for name, chunk_list in strategies.items():
    print(f"=== {name} ===")
    hits = [c for c in chunk_list if "45일" in c]
    if hits:
        print(hits[0][:200])
    print()

=== 고정 크기 ===
시작일이 포함된 사업연도 - 사업 종료일을 초과할 수 없다.

### 제12조(재계약)

① 전임입학사정관은 제2조 1항의 입학사정관제 지원 사업을 계속하는 경우, 그 업무 추진실적과 근무성적 평가결과에 따라 재계약을 통하여 계약기간을 연장할 수 있다.

② 계약기간을 연장하고자 하는 전임입학사정관은 계약기간 만료 45일 전까지 별지4의 ‘재계약 신청서’를

=== 재귀적 ===
② 계약기간을 연장하고자 하는 전임입학사정관은 계약기간 만료 45일 전까지 별지4의 ‘재계약 신청서’를 본부장에게 제출하여야 한다.

③ 전임입학사정관으로부터 재계약 신청이 접수되면 총장은 인사위원회를 소집하여 심사절차를 진행하고, 심사결과를 계약기간 만료 1개월 전까지 본인에게 통보한다.

### 제13조(근무성적 평가)

① 전임입학사정관의 근무성적 평

=== 구조 기반 ===
① 전임입학사정관은 제2조 1항의 입학사정관제 지원 사업을 계속하는 경우, 그 업무 추진실적과 근무성적 평가결과에 따라 재계약을 통하여 계약기간을 연장할 수 있다.  
② 계약기간을 연장하고자 하는 전임입학사정관은 계약기간 만료 45일 전까지 별지4의 ‘재계약 신청서’를 본부장에게 제출하여야 한다.  
③ 전임입학사정관으로부터 재계약 신청이 접수되면 총장은

=== 시맨틱 ===
1. 위촉 협약서(소정 양식)

2. 서약서(별지1의 서식)

3. 이력서(별지2의 서식)

4. 자기소개서(별지3의 서식)

5. 경력증명서(해당자에 한함)

② 전임입학사정관의 임용시 구비서류에 대해서는 별도로 정한다. ### 제11조(계약기간)

① 전임입학사정관의 임용기간은 1년으로 하며, 제2조 1항의 입학사정관제 지원 사업을 계속하는 경우 그 기



`chunk_size`와 `chunk_overlap`을 바꿔가며 재귀적 청킹이 어떻게 변하는지도 표로 확인합니다. (강의안의 실습을 표로 정리한 것입니다.)

In [ ]:
# 강의안의 실습(chunk_size, chunk_overlap 바꿔 보기)을 반복문으로 한 번에 실행합니다
rows = []
for size, overlap in [(150, 0), (300, 50), (500, 100), (800, 150)]:
    sp = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=size,
        chunk_overlap=overlap,
    )
    cs = sp.split_text(document)
    rows.append({
        "chunk_size": size, "overlap": overlap, "청크 수": len(cs),
        "평균 길이": round(sum(len(c) for c in cs) / len(cs)),
        "조 보존율": f"{intact_ratio(cs):.0%}", "조 혼합 청크": mixed_count(cs),
    })
display(pd.DataFrame(rows))

,chunk_size,overlap,청크 수,평균 길이,조 보존율,조 혼합 청크
0,150,0,50,112,20%,1
1,300,50,23,264,48%,5
2,500,100,15,439,80%,12
3,800,150,9,722,88%,8


### 청킹 전략 선택과 그 이유 : **문서 구조 기반(조 단위)**

1. **문서의 논리 단위가 조(條)입니다.** 규정 문서는 "제N조" 하나가 하나의 주제이고, 사용자의 질문도 "제N조에 따르면"처럼 조 단위로 근거를 요구합니다. 구조 기반 청킹은 이 단위를 그대로 청크로 만듭니다.
2. **조가 다른 조와 섞이거나 중간에서 잘리는 일이 없습니다.** 위 표에서 고정 크기·재귀적 청킹은 조 보존율이 100% 미만이거나 여러 조가 한 청크에 섞이는데, 구조 기반은 조 보존율 100%입니다. 시맨틱 청킹은 청크 수가 적고 길이 편차가 커서, 서로 다른 조가 한 덩어리로 묶이기 쉽습니다.
3. **길이가 모델 상한 안에 들어옵니다.** 가장 긴 청크의 토큰 수가 위 출력에서 E5 상한(512)보다 작아서, 조를 더 쪼갤 필요가 없고 뒷부분이 잘리는 위험도 없습니다.
4. **메타데이터(장·조 이름)가 자동으로 붙습니다.** 답변의 근거 조항 표기와 장(章) 단위 필터 검색에 그대로 쓸 수 있습니다.

이후 단계에서는 구조 기반 청킹의 결과(`md_chunks`)를 사용합니다. 다만 구조 기반 청킹에서는 조 제목이 메타데이터로 빠져서 **청크 본문에 "제17조"라는 글자가 없습니다.** 이것이 식별자 질문("제17조가 뭐야?")에서 약점이 될 수 있어서, Step 7의 개선 실험에서 다룹니다.

## Step 3 : 오픈소스 임베딩 비교 실험

첫 번째 취약점(임베딩 API로 문서 전문이 나가는 문제)을 막습니다. BGE-M3와 multilingual-e5를 로드하고, 이 문서에서 어느 쪽이 검색을 더 잘하는지 직접 측정합니다.

모델을 비교하려면 기준이 필요합니다. **질문과 정답 조항의 쌍**으로 평가 세트를 만듭니다. 정답을 청크 번호 대신 **조 이름**으로 적고, 번호는 코드가 변환하게 했습니다. 손으로 번호를 세다 틀리는 일을 막기 위해서입니다.

In [ ]:
# [변경] 청크 번호 ↔ 조 이름 대응표를 만듭니다. "제7조 (…)"나 "부칙 4" 같은 이름에서 짧은 이름표만 뽑습니다
chunks = md_chunks

def article_label(metadata):
    return re.match(r"제\d+조|부칙 \d+", metadata["조항"]).group()   # 예 : "제17조(위원장)" → "제17조"

chunk_ids = [article_label(c.metadata) for c in chunks]              # 청크 번호 순서대로 나열한 조 이름표
id_to_idx = {label: i for i, label in enumerate(chunk_ids)}          # 조 이름 → 청크 번호

# 청크 목록을 번호와 함께 확인합니다
for i, c in enumerate(chunks):
    print(f"[{i:>2}] {c.metadata['조항']}   ({c.metadata['장']})")

[ 0] 제1조(목적)   (제1장 총칙)
[ 1] 제2조(입학사정관)   (제1장 총칙)
[ 2] 제3조(사정관제 전형)   (제1장 총칙)
[ 3] 제4조(사정원칙)   (제1장 총칙)
[ 4] 제5조(입학사정관실)   (제2장 기구 및 업무)
[ 5] 제6조(입학사정관의 업무)   (제2장 기구 및 업무)
[ 6] 제7조(입학사정관의 수)   (제3장 입학사정관의 임용 및 위촉)
[ 7] 제8조(입학사정관의 임용 및 위촉)   (제3장 입학사정관의 임용 및 위촉)
[ 8] 제9조(자격 기준)   (제3장 입학사정관의 임용 및 위촉)
[ 9] 제10조(구비서류)   (제3장 입학사정관의 임용 및 위촉)
[10] 제11조(계약기간)   (제3장 입학사정관의 임용 및 위촉)
[11] 제12조(재계약)   (제3장 입학사정관의 임용 및 위촉)
[12] 제13조(근무성적 평가)   (제3장 입학사정관의 임용 및 위촉)
[13] 제14조(인사위원회)   (제4장 인사위원회)
[14] 제15조(구성 및 임기)   (제4장 인사위원회)
[15] 제16조(위원장)   (제4장 인사위원회)
[16] 제17조(위원회의 소집 및 운영)   (제4장 인사위원회)
[17] 제18조(소위원회)   (제4장 인사위원회)
[18] 제19조(입학사정관의 활동 및 교육)   (제5장 기타)
[19] 제20조(위임전결)   (제5장 기타)
[20] 제21조(입학사정관의 취업제한 등)   (제5장 기타)
[21] 부칙 1   (부칙)
[22] 부칙 2   (부칙)
[23] 부칙 3   (부칙)
[24] 부칙 4   (부칙)


평가 질문은 총 **18개**입니다. 의미 질문 12개는 규정 표현을 그대로 쓰지 않고 일상적인 말로 바꿔 썼습니다(예 : "취업제한" → "학원에서 일할 수 있어?"). 식별자 질문 6개는 조 번호, 별지 번호, 날짜처럼 **정확한 글자 일치가 필요한 질문**입니다. 정답은 여러 조가 관련되더라도 **가장 직접적인 근거 조 1개**로 정했습니다.

In [ ]:
# 평가 세트 : (질문, 정답 조항, 질문 유형)
eval_set = [
    # ---- 의미 질문 12개 : 규정에 나온 표현과 다르게 물어봅니다 ----
    ("입학사정관은 어떤 종류가 있어?",                       "제2조",  "의미"),
    ("전임 입학사정관 계약은 몇 년이야?",                     "제11조", "의미"),
    ("재계약하려면 언제까지 신청해야 해?",                    "제12조", "의미"),
    ("인사위원회는 몇 명으로 구성돼?",                        "제15조", "의미"),
    ("위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?",      "제17조", "의미"),
    ("입학사정관을 그만둔 뒤에 학원에서 일할 수 있어?",        "제21조", "의미"),
    ("입학사정관이 되려면 학위가 필요해?",                    "제9조",  "의미"),
    ("초빙입학사정관을 위촉할 때 내야 하는 서류는?",           "제10조", "의미"),
    ("위원장이 자리를 비우면 누가 대신해?",                   "제16조", "의미"),
    ("소위원회 위원장은 누구야?",                            "제18조", "의미"),
    ("전임입학사정관 근무성적은 누가 평가해?",                 "제13조", "의미"),
    ("입학사정관이 하는 일은 뭐야?",                         "제6조",  "의미"),
    # ---- 식별자 질문 6개 : 조 번호, 별지 번호, 날짜 등 정확한 글자가 필요합니다 ----
    ("제17조가 무슨 내용이야?",                              "제17조", "식별자"),
    ("제19조 알려줘",                                       "제19조", "식별자"),
    ("제11조 3항에 나온 초빙입학사정관 위촉기간 한도는?",       "제11조", "식별자"),
    ("별지4는 어떤 서식이야?",                               "제12조", "식별자"),
    ("2013년 5월 28일 개정은 언제부터 효력이 생겨?",           "부칙 4", "식별자"),
    ("별지1의2 서약서는 언제 제출해?",                         "제21조", "식별자"),
]

# 정답 조항 이름을 청크 번호로 바꿉니다 (강의안의 eval_set 과 같은 모양이 됩니다)
eval_idx = [(q, id_to_idx[gold], qtype) for q, gold, qtype in eval_set]

print(f"평가 질문 수 : {len(eval_set)}개 (의미 {sum(t == '의미' for _, _, t in eval_set)}개, 식별자 {sum(t == '식별자' for _, _, t in eval_set)}개)")

평가 질문 수 : 18개 (의미 12개, 식별자 6개)


In [ ]:
# 두 모델을 로드합니다. BGE-M3는 처음 실행 시 약 2.2GB를 내려받습니다

bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},   # 코사인 유사도 계산을 위한 정규화
)

e5_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
# 임베딩 모델이 얼마나 가벼운지 확인해봅시다
print(f"현재 VRAM 사용량 : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

현재 VRAM 사용량 : 3.86 GB


In [ ]:
chunk_texts = [c.page_content for c in chunks]

# BGE-M3 : 그대로 임베딩합니다
bge_doc_vecs = np.array(bge_embeddings.embed_documents(chunk_texts))

# E5 : 문서에는 passage prefix를 붙입니다 (E5의 사용 규칙)
e5_doc_vecs = np.array(
    e5_embeddings.embed_documents([f"passage: {t}" for t in chunk_texts])
)

print(f"BGE-M3 문서 벡터 : {bge_doc_vecs.shape}")
print(f"E5 문서 벡터     : {e5_doc_vecs.shape}")

# [변경] 트랙 3의 진단 습관 : 벡터 길이가 1인지 확인합니다 (정규화 누락은 에러 없이 성능만 깎는 실패입니다)
print(f"BGE-M3 벡터 길이 : {np.linalg.norm(bge_doc_vecs, axis=1).min():.3f} ~ {np.linalg.norm(bge_doc_vecs, axis=1).max():.3f}")
print(f"E5 벡터 길이     : {np.linalg.norm(e5_doc_vecs, axis=1).min():.3f} ~ {np.linalg.norm(e5_doc_vecs, axis=1).max():.3f}")

BGE-M3 문서 벡터 : (25, 1024)
E5 문서 벡터     : (25, 768)
BGE-M3 벡터 길이 : 1.000 ~ 1.000
E5 벡터 길이     : 1.000 ~ 1.000


벡터의 차원 수가 다릅니다. BGE-M3는 1024차원, e5-base는 768차원입니다. 벡터DB 컬렉션을 만들 때 반드시 맞춰야 하는 값입니다. 벡터 길이는 둘 다 1.000이어야 정규화가 제대로 된 것입니다.

평가 지표는 강의안과 같은 **Hit@k**입니다. 정답 조가 검색 상위 k개 안에 있으면 성공으로 칩니다.

In [ ]:
def hit_at_k(query_vec, doc_vecs, gold_idx, k=3):
    """정답 청크가 상위 k개 안에 있으면 1, 없으면 0"""
    scores = doc_vecs @ query_vec            # 정규화된 벡터이므로 내적 = 코사인 유사도
    top_k = np.argsort(scores)[::-1][:k]
    return int(gold_idx in top_k), top_k


# [변경] 강의안의 evaluate 에 "질문 유형별 Hit@3"과 "틀린 질문의 top3 조 이름 출력"을 추가했습니다
def evaluate(name, embeddings, doc_vecs, query_prefix=""):
    hits_1, hits_3 = 0, 0
    by_type = {"의미": [0, 0], "식별자": [0, 0]}      # 유형별 [맞힌 수, 전체 수] (Hit@3 기준)
    print(f"=== {name} ===")
    for q, gold, qtype in eval_idx:
        q_vec = np.array(embeddings.embed_query(query_prefix + q))
        h1, _ = hit_at_k(q_vec, doc_vecs, gold, k=1)
        h3, top3 = hit_at_k(q_vec, doc_vecs, gold, k=3)
        hits_1 += h1
        hits_3 += h3
        by_type[qtype][0] += h3
        by_type[qtype][1] += 1
        mark = "O" if h3 else "X"
        print(f"  [{mark}] ({qtype}) {q}  →  top3 : {[chunk_ids[i] for i in top3]}, 정답 : {chunk_ids[gold]}")
    n = len(eval_idx)
    print(f"  Hit@1 : {hits_1}/{n} ({hits_1/n:.0%})   Hit@3 : {hits_3}/{n} ({hits_3/n:.0%})")
    print(f"  Hit@3 유형별 : 의미 {by_type['의미'][0]}/{by_type['의미'][1]}, 식별자 {by_type['식별자'][0]}/{by_type['식별자'][1]}\n")
    return hits_1 / n, hits_3 / n

In [ ]:
# E5는 질문에 query prefix를 붙입니다 (E5의 사용 규칙)
bge_scores = evaluate("BGE-M3", bge_embeddings, bge_doc_vecs)
e5_scores = evaluate("multilingual-e5-base", e5_embeddings, e5_doc_vecs, query_prefix="query: ")

=== BGE-M3 ===
  [O] (의미) 입학사정관은 어떤 종류가 있어?  →  top3 : ['제2조', '제5조', '제6조'], 정답 : 제2조
  [O] (의미) 전임 입학사정관 계약은 몇 년이야?  →  top3 : ['제11조', '제12조', '제2조'], 정답 : 제11조
  [O] (의미) 재계약하려면 언제까지 신청해야 해?  →  top3 : ['제12조', '제11조', '제13조'], 정답 : 제12조
  [O] (의미) 인사위원회는 몇 명으로 구성돼?  →  top3 : ['제15조', '제14조', '제8조'], 정답 : 제15조
  [O] (의미) 위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?  →  top3 : ['제17조', '제15조', '제18조'], 정답 : 제17조
  [O] (의미) 입학사정관을 그만둔 뒤에 학원에서 일할 수 있어?  →  top3 : ['제21조', '제2조', '제6조'], 정답 : 제21조
  [O] (의미) 입학사정관이 되려면 학위가 필요해?  →  top3 : ['제9조', '제2조', '제6조'], 정답 : 제9조
  [O] (의미) 초빙입학사정관을 위촉할 때 내야 하는 서류는?  →  top3 : ['제10조', '제8조', '제9조'], 정답 : 제10조
  [O] (의미) 위원장이 자리를 비우면 누가 대신해?  →  top3 : ['제16조', '제15조', '제18조'], 정답 : 제16조
  [O] (의미) 소위원회 위원장은 누구야?  →  top3 : ['제18조', '제16조', '제15조'], 정답 : 제18조
  [O] (의미) 전임입학사정관 근무성적은 누가 평가해?  →  top3 : ['제13조', '제8조', '제7조'], 정답 : 제13조
  [O] (의미) 입학사정관이 하는 일은 뭐야?  →  top3 : ['제6조', '제5조', '제2조'], 정답 : 제6조
  [O] (식별자) 제17조가 무슨 내용이야?  →  top3 : ['제18조', '제17조', 

In [ ]:
# 트랙 3의 "실패 2" 확인 : E5를 prefix 없이 평가해봅시다 (문서 벡터도 prefix 없이 다시 만들어야 공정한 비교입니다)
e5_doc_vecs_noprefix = np.array(
    e5_embeddings.embed_documents(chunk_texts)      # passage prefix 없이
)

e5_no_prefix_scores = evaluate("E5 (prefix 없음)", e5_embeddings, e5_doc_vecs_noprefix)   # query prefix도 없이

=== E5 (prefix 없음) ===
  [X] (의미) 입학사정관은 어떤 종류가 있어?  →  top3 : ['제5조', '제6조', '제14조'], 정답 : 제2조
  [O] (의미) 전임 입학사정관 계약은 몇 년이야?  →  top3 : ['제11조', '제12조', '제21조'], 정답 : 제11조
  [O] (의미) 재계약하려면 언제까지 신청해야 해?  →  top3 : ['제12조', '제11조', '제21조'], 정답 : 제12조
  [O] (의미) 인사위원회는 몇 명으로 구성돼?  →  top3 : ['제14조', '제15조', '제17조'], 정답 : 제15조
  [O] (의미) 위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?  →  top3 : ['제17조', '제15조', '제18조'], 정답 : 제17조
  [O] (의미) 입학사정관을 그만둔 뒤에 학원에서 일할 수 있어?  →  top3 : ['제21조', '제12조', '제2조'], 정답 : 제21조
  [O] (의미) 입학사정관이 되려면 학위가 필요해?  →  top3 : ['제9조', '제21조', '제6조'], 정답 : 제9조
  [O] (의미) 초빙입학사정관을 위촉할 때 내야 하는 서류는?  →  top3 : ['제10조', '제9조', '제2조'], 정답 : 제10조
  [O] (의미) 위원장이 자리를 비우면 누가 대신해?  →  top3 : ['제16조', '제17조', '제18조'], 정답 : 제16조
  [O] (의미) 소위원회 위원장은 누구야?  →  top3 : ['제18조', '제17조', '제16조'], 정답 : 제18조
  [O] (의미) 전임입학사정관 근무성적은 누가 평가해?  →  top3 : ['제13조', '제9조', '제8조'], 정답 : 제13조
  [O] (의미) 입학사정관이 하는 일은 뭐야?  →  top3 : ['제6조', '제5조', '제2조'], 정답 : 제6조
  [O] (식별자) 제17조가 무슨 내용이야?  →  top3 : ['제1

**기준선(baseline)의 임베딩 모델은 BGE-M3로 고정합니다.** 강의안의 기본 선택이고, 측정 결과를 본 뒤에 유리한 모델로 바꾸면 기준선이 "결과에 맞춘 값"이 되어 개선 실험과 비교할 수 없기 때문입니다. E5가 더 높게 나왔더라도 기준선은 BGE-M3로 두고, E5는 Step 7의 개선 실험(② 임베딩 모델 교체)에서 다시 비교합니다.

측정 결과 BGE-M3는 Hit@1 78%·Hit@3 89%, E5는 Hit@1 67%·Hit@3 78%였습니다. 이 문서에서는 처음부터 BGE-M3가 더 높았으므로, 결과를 보고 모델을 바꾼 것이 아니라 강의안의 기본 선택을 그대로 유지한 것입니다. 식별자 질문의 실패(제19조, 별지4)는 두 모델 모두에서 나타났습니다. 즉 모델의 차이보다 청크에 번호 글자가 없다는 점이 원인일 가능성이 큽니다. (트랙 3에서 확인한 벡터 길이 1.000, 최대 토큰 248 < 상한 512도 정상이라 정규화 누락이나 512토큰 잘림은 이 문서에서 일어나지 않았습니다.)

## Step 4 : Qdrant 로컬 세팅

두 번째 취약점(매니지드 벡터DB에 벡터와 원문이 저장되는 문제)을 막습니다. Qdrant를 임베디드 모드로 시작하면 서버 설치 없이 로컬 폴더에 데이터가 파일로 저장됩니다.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# path를 지정하면 임베디드 모드입니다. 해당 폴더에 데이터가 저장됩니다
client = QdrantClient(path="./qdrant_local")

나중에 서버 모드로 옮길 때는 이 한 줄만 바뀝니다. `client = QdrantClient(url="http://our-internal-server:6333")`

**컬렉션**은 관계형 DB의 테이블에 해당하고, 벡터 차원 수와 거리 계산 방식을 지정해야 합니다. 임베딩 모델과 컬렉션은 한 몸입니다.

In [ ]:
# 컬렉션 생성. 관계형 DB의 테이블에 해당합니다
# 벡터 차원 수와 거리 계산 방식을 지정해야 합니다

COLLECTION = "hongik_regulation"    # [변경] 이 프로젝트의 문서에 맞춘 이름

# 재실행에 대비해, 이미 있으면 지우고 새로 만듭니다
# (recreate_collection은 폐기 예정 API라 쓰지 않습니다)
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=1024,                  # BGE-M3의 차원 수. Step 3에서 확인한 값입니다
        distance=Distance.COSINE,
    ),
)

True

In [ ]:
from qdrant_client.models import PointStruct

# 벡터와 함께 payload(메타데이터)를 저장합니다
# Step 2의 구조 기반 청킹이 만들어준 장·조항 정보가 여기서 쓰입니다

points = [
    PointStruct(
        id=i,
        vector=bge_doc_vecs[i].tolist(),
        payload={
            "text": chunks[i].page_content,
            "장": chunks[i].metadata.get("장", ""),
            "조항": chunks[i].metadata.get("조항", ""),
            "id_label": chunk_ids[i],      # [변경] 정답 채점에 쓰는 짧은 조 이름표 ("제17조", "부칙 4")
        },
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION, points=points)
print(f"{len(points)}개 포인트 저장 완료")

25개 포인트 저장 완료


In [ ]:
# 기본 검색을 해봅니다

query = "입학사정관이 퇴직하면 학원에 취업할 수 있나요?"   # [변경] 이 문서에 맞는 질문
query_vec = bge_embeddings.embed_query(query)

results = client.query_points(
    collection_name=COLLECTION,
    query=query_vec,
    limit=3,
).points

for r in results:
    print(f"[유사도 {r.score:.3f}] {r.payload['조항']}")
    print(f"  {r.payload['text'][:70]}...")

[유사도 0.782] 제21조(입학사정관의 취업제한 등)
  ① 입학사정관은 퇴직한 날 이후 3년 동안 「학원의 설립·운영 및 과외교습에 관한 법률」 제2조제1호에 따른 학원을 설립하거나...
[유사도 0.559] 제2조(입학사정관)
  ① 교육과학기술부의 입학사정관제 지원사업(기간제 및 단시간근로자 보호 등에 관한 법률 제4조 1항 1호의 사업을 의미한다)을 ...
[유사도 0.552] 제9조(자격 기준)
  ① 전임입학사정관과 초빙입학사정관은 다음 각 호의 자격기준을 갖추어야 한다.  
1. 국내․외 4년제 대학 학사학위 이상 소지...


이번에는 Qdrant의 특기인 **payload 필터**입니다. "제4장 인사위원회에 해당하는 조 중에서만 검색"을 해봅니다.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

results = client.query_points(
    collection_name=COLLECTION,
    query=bge_embeddings.embed_query("위원장이 자리에 없으면 누가 대신해?"),
    query_filter=Filter(
        must=[FieldCondition(key="장", match=MatchValue(value="제4장 인사위원회"))]
    ),
    limit=3,
).points

for r in results:
    print(f"[유사도 {r.score:.3f}] {r.payload['조항']}")

[유사도 0.666] 제16조(위원장)
[유사도 0.545] 제15조(구성 및 임기)
[유사도 0.527] 제18조(소위원회)


Step 2에서 "구조 기반 청킹의 메타데이터를 나중에 쓴다"고 했던 것이 여기서 쓰였습니다. 청킹 단계에서 남긴 구조 정보가 검색 단계의 필터가 됩니다.

In [ ]:
# 프라이빗 관점의 확인 : 우리 벡터는 지금 어디에 있을까요?

!ls -lh ./qdrant_local/collection/hongik_regulation/

total 276K
-rw-r--r-- 1 root root 272K Sep 30 01:10 storage.sqlite


로컬 폴더의 파일입니다. 클라우드 어디에도 없습니다. 이 폴더를 지우면 데이터가 사라집니다. (Step 9의 오프라인 검증에서도 이 폴더의 컬렉션을 그대로 씁니다.)

## Step 5 : 로컬 LLM 로드

마지막 세 번째 취약점(LLM API로 질문과 검색된 문서가 나가는 문제)을 막습니다. 생성을 담당하는 LLM을 로컬 GPU에 올립니다. 모델은 **Qwen2.5-7B-Instruct**이고, T4에서 fp16으로 올리면 약 14GB라 여유가 없으므로 **4bit 양자화**로 로드합니다.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                  # 4bit 표현 방식
    bnb_4bit_compute_dtype=torch.float16,       # 계산 정밀도. T4는 fp16 (bf16 미지원)
    bnb_4bit_use_double_quant=True,             # 양자화 상수를 한 번 더 양자화해 메모리 절약
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,      # 양자화되지 않는 계층(임베딩 등)의 정밀도. T4는 bf16 미지원이라 fp16을 명시합니다
)
model.eval()

print(f"모델 로드 완료. VRAM 사용량 : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

모델 로드 완료. VRAM 사용량 : 9.45 GB


처음 실행하면 모델 파일 약 15GB를 내려받아 시간이 걸립니다. 로드 후 VRAM은 LLM 약 5.5GB에 임베딩 모델(BGE-M3, E5-base)이 더해져 **8~10GB 수준이면 정상**입니다. 뒤에서 리랭커(약 2.2GB)를 추가로 올립니다.

In [ ]:
# 생성 함수를 만듭니다
# Instruct 모델은 학습 때 정해진 대화 형식이 있어, 그 형식대로 입력해야 제 성능이 나옵니다
# apply_chat_template이 그 형식을 자동으로 맞춰줍니다

def generate(system_prompt: str, user_prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,         # input_ids 와 attention_mask 를 함께 받습니다
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,          # 사실 기반 답변이므로 낮게 설정합니다
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# 동작 확인
print(generate("당신은 친절한 비서입니다.", "안녕하세요, 자기소개 해주세요.", max_new_tokens=100))

안녕하세요! 저는 당신의 비서로 활동하고 있는 인공지능 비서입니다. 다양한 정보 제공, 일정 관리, 질문에 대한 답변 등 다양한 업무를 도와드릴 수 있습니다. 항상 친절하고 정확하게 정보를 제공하기 위해 노력하겠습니다. 오늘도 좋은 하루 보내세요!


## Step 6 : 완성 예제, 완전 로컬 RAG

로컬 임베딩, 로컬 벡터DB, 로컬 LLM을 연결해 RAG를 완성하고, 이것을 **기준선(baseline)** 으로 삼아 Hit@3을 측정합니다.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list:
    """로컬 임베딩 + 로컬 Qdrant 검색"""
    query_vec = bge_embeddings.embed_query(query)
    results = client.query_points(
        collection_name=COLLECTION,
        query=query_vec,
        limit=top_k,
    ).points
    return [
        # [변경] 정답 채점을 위해 조 이름표(id_label)도 함께 돌려줍니다
        {"조항": r.payload["조항"], "내용": r.payload["text"], "유사도": r.score, "id_label": r.payload["id_label"]}
        for r in results
    ]

In [ ]:
# [변경] 이 프로젝트의 문서에 맞게 바꾸고, 강의안이 알려준 4번 규칙(한국어 답변)을 처음부터 넣었습니다
SYSTEM_PROMPT = """당신은 홍익대학교 입학사정관 규정 안내 챗봇입니다.
아래 규칙을 지키세요.
1. 제공된 규정 내용에 근거해서만 답변합니다.
2. 답변 끝에 근거 조항을 표기합니다.
3. 규정에 없는 내용이면 "규정에서 확인되지 않습니다"라고 답합니다.
4. 답변은 반드시 한국어로 작성합니다."""


def rag_answer(query: str, top_k: int = 3) -> str:
    contexts = retrieve(query, top_k)

    context_text = "\n\n".join(
        f"[{c['조항']}]\n{c['내용']}" for c in contexts
    )
    user_prompt = f"""다음은 규정에서 검색된 내용입니다.

{context_text}

질문 : {query}"""

    return generate(SYSTEM_PROMPT, user_prompt)

In [ ]:
print(rag_answer("전임입학사정관 계약은 몇 년이고, 재계약은 언제까지 신청해야 해?"))

전임입학사정관의 임용기간은 1년이며, 그 기간 내에서 연장할 수 있습니다. 재계약을 원하는 전임입학사정관은 계약기간 만료 45일 전까지 '재계약 신청서'를 본부장에게 제출해야 합니다.

- 제11조 ①항
- 제12조 ②항


답변이 제11조(계약기간)와 제12조(재계약)를 근거로 나오는지 확인해 주세요. 두 조에 걸친 질문이라 검색이 두 조를 모두 가져와야 하고, 이것이 이 문서의 어려운 점 중 하나입니다.

In [ ]:
# 몇 가지 질문을 더 던져봅니다

questions = [
    "입학사정관이 퇴직하고 나서 입시학원에 취업해도 돼?",
    "위원회 회의록에는 뭐가 필요해?",
    "기숙사 신청은 어떻게 해?",           # 규정에 없는 질문입니다
]

for q in questions:
    print(f"Q : {q}")
    print(f"A : {rag_answer(q)}")
    print("=" * 60)

Q : 입학사정관이 퇴직하고 나서 입시학원에 취업해도 돼?
A : 입학사정관은 퇴직한 날 이후 3년 동안 「학원의 설립·운영 및 과외교습에 관한 법률」 제2조제1호에 따른 학원을 설립하거나 이에 취업할 수 없습니다. 하지만 「교육공무원법」 제5조에 따른 인사위원회 또는 「사립학교법」 제53조의3에 따른 교원인사위원회의 승인을 받은 경우에는 그러하지 않을 수 있습니다.

따라서 일반적으로는 퇴직 후 3년 동안 입시학원에 취업할 수 없으나, 특정 조건 하에서는 가능합니다.

참조 조항: 제21조(입학사정관의 취업제한 등) ①
Q : 위원회 회의록에는 뭐가 필요해?
A : 위원회 회의록에는 회의의 모든 사항이 기재되어야 하며, 이를 위해 위원장의 서명 또는 날인을 받아야 합니다. 이는 제17조 제6항에 근거합니다.
Q : 기숙사 신청은 어떻게 해?
A : 규정에서 확인되지 않습니다.


마지막 질문이 중요합니다. 규정에 없는 내용에 대해 "확인되지 않는다"고 답하는지 보세요. 지어내서 답한다면 시스템 프롬프트를 더 단단하게 다듬어야 합니다. 이 방어를 검색 점수로도 할 수 있는지, 강의안의 "조기 반환"(유사도가 기준 미만이면 LLM을 부르지 않음)을 확인합니다. 기준값은 **점수 분포를 먼저 보고** 정해야 합니다.

In [ ]:
# 규정 안의 질문과 규정 밖의 질문에서 "1등 조항의 유사도"가 얼마나 다른지 확인합니다
in_doc = [q for q, _, _ in eval_set[:5]]
out_doc = ["기숙사 신청은 어떻게 해?", "학교 식당 메뉴 알려줘", "장학금은 언제 지급돼?"]

for label, qs in [("규정 안", in_doc), ("규정 밖", out_doc)]:
    for q in qs:
        print(f"[{label}] 1등 유사도 {retrieve(q, 1)[0]['유사도']:.3f}   {q}")

[규정 안] 1등 유사도 0.663   입학사정관은 어떤 종류가 있어?
[규정 안] 1등 유사도 0.702   전임 입학사정관 계약은 몇 년이야?
[규정 안] 1등 유사도 0.645   재계약하려면 언제까지 신청해야 해?
[규정 안] 1등 유사도 0.611   인사위원회는 몇 명으로 구성돼?
[규정 안] 1등 유사도 0.571   위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?
[규정 밖] 1등 유사도 0.518   기숙사 신청은 어떻게 해?
[규정 밖] 1등 유사도 0.430   학교 식당 메뉴 알려줘
[규정 밖] 1등 유사도 0.481   장학금은 언제 지급돼?


In [ ]:
# 강의안의 조기 반환 실습 : 검색된 조항의 유사도가 기준 미만이면 LLM을 호출하지 않고 바로 반환합니다
# [변경] threshold 는 위의 점수 분포를 보고 정해야 합니다 (규정 안 질문과 규정 밖 질문의 점수가 겹치면 기준값 하나로는 못 거릅니다)
def rag_answer_v2(query: str, top_k: int = 3, threshold: float = 0.4) -> str:
    contexts = retrieve(query, top_k)

    # 조기 반환 : 가장 유사한 조항조차 기준 미달이면 LLM을 부르지 않습니다
    if contexts[0]["유사도"] < threshold:
        return "규정에서 확인되지 않습니다."

    context_text = "\n\n".join(f"[{c['조항']}]\n{c['내용']}" for c in contexts)
    user_prompt = f"다음은 규정에서 검색된 내용입니다.\n\n{context_text}\n\n질문 : {query}"
    return generate(SYSTEM_PROMPT, user_prompt)


print(rag_answer_v2("기숙사 신청은 어떻게 해?"))

규정에서 확인되지 않습니다.


위 점수 분포를 보면 규정 안 질문(5개 표본)의 1등 유사도는 0.571~0.702, 규정 밖 질문은 0.430~0.518이었습니다. 코드의 기본값 `threshold=0.4`는 규정 밖 질문 3개보다 모두 낮은 값이라 **어느 질문도 걸러내지 못합니다**. 실제로 "기숙사 신청은 어떻게 해?"(0.518)는 조기 반환이 아니라 LLM 프롬프트의 규칙에 의해 "규정에서 확인되지 않습니다."가 나온 것입니다. 이 표본에서는 0.52~0.57 사이에 값을 두면 안팎이 갈리지만, 두 집단의 간격이 0.05 안팎으로 좁고 규정 안 질문에서 낮은 점수(예: 일상어로 바꾼 질문)가 나올 수 있어서, 점수 하나로 거르는 방식은 위험하다고 판단했습니다. 그래서 이 프로젝트에서는 **검색 점수 기준보다 LLM 프롬프트의 규칙(규정에 없으면 "확인되지 않습니다")을 방어선으로 쓰고**, `threshold`는 참고용으로 남겨 둡니다.

### 기준선 Hit@3 측정

Step 3에서는 벡터를 직접 곱해서 비교했지만, 여기서는 **실제 파이프라인(`retrieve` : 로컬 임베딩 + Qdrant)** 으로 18개 질문을 검색해서 측정합니다. 이 값이 개선 실험의 기준선입니다.

In [ ]:
# [변경] 강의안의 dense_only_hit3 를 "실제 파이프라인 + 질문 유형별 + 질문별 출력"으로 확장했습니다
def measure_baseline(top_k: int = 3):
    by_type = {"의미": [0, 0], "식별자": [0, 0]}          # 유형별 [맞힌 수, 전체 수]
    total = 0
    print("=== 기준선 : 조 본문 + BGE-M3 + Qdrant, 밀집 검색만 ===")
    for q, gold_label, qtype in eval_set:
        found = [c["id_label"] for c in retrieve(q, top_k)]     # 상위 top_k개의 조 이름표
        ok = gold_label in found
        total += ok
        by_type[qtype][0] += ok
        by_type[qtype][1] += 1
        print(f"  [{'O' if ok else 'X'}] ({qtype}) {q}  →  {found}, 정답 : {gold_label}")
    n = len(eval_set)
    print(f"\n기준선 Hit@{top_k} : {total}/{n} ({total/n:.0%})")
    print(f"  의미 질문   : {by_type['의미'][0]}/{by_type['의미'][1]}")
    print(f"  식별자 질문 : {by_type['식별자'][0]}/{by_type['식별자'][1]}")
    return total / n

baseline_hit3 = measure_baseline()

=== 기준선 : 조 본문 + BGE-M3 + Qdrant, 밀집 검색만 ===
  [O] (의미) 입학사정관은 어떤 종류가 있어?  →  ['제2조', '제5조', '제6조'], 정답 : 제2조
  [O] (의미) 전임 입학사정관 계약은 몇 년이야?  →  ['제11조', '제12조', '제2조'], 정답 : 제11조
  [O] (의미) 재계약하려면 언제까지 신청해야 해?  →  ['제12조', '제11조', '제13조'], 정답 : 제12조
  [O] (의미) 인사위원회는 몇 명으로 구성돼?  →  ['제15조', '제14조', '제8조'], 정답 : 제15조
  [O] (의미) 위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?  →  ['제17조', '제15조', '제18조'], 정답 : 제17조
  [O] (의미) 입학사정관을 그만둔 뒤에 학원에서 일할 수 있어?  →  ['제21조', '제2조', '제6조'], 정답 : 제21조
  [O] (의미) 입학사정관이 되려면 학위가 필요해?  →  ['제9조', '제2조', '제6조'], 정답 : 제9조
  [O] (의미) 초빙입학사정관을 위촉할 때 내야 하는 서류는?  →  ['제10조', '제8조', '제9조'], 정답 : 제10조
  [O] (의미) 위원장이 자리를 비우면 누가 대신해?  →  ['제16조', '제15조', '제18조'], 정답 : 제16조
  [O] (의미) 소위원회 위원장은 누구야?  →  ['제18조', '제16조', '제15조'], 정답 : 제18조
  [O] (의미) 전임입학사정관 근무성적은 누가 평가해?  →  ['제13조', '제8조', '제7조'], 정답 : 제13조
  [O] (의미) 입학사정관이 하는 일은 뭐야?  →  ['제6조', '제5조', '제2조'], 정답 : 제6조
  [O] (식별자) 제17조가 무슨 내용이야?  →  ['제18조', '제17조', '제21조'], 정답 : 제17조
  [X] (식별자) 제19조 알려줘  →  ['제20조', '제18조', 

기준선을 기록했습니다. 어떤 질문에서 틀렸는지, 의미 질문과 식별자 질문 중 어느 쪽이 약한지를 위 출력에서 확인해 주세요. 이 관찰이 Step 7의 실험 설계 근거입니다.

## Step 7 : 개선 실험

기준선에서 **한 번에 하나만** 바꿔서 Hit@3이 어떻게 달라지는지 봅니다. 트랙 3(진단)과 트랙 5(리랭커)에서 배운 기법을 이 문서에 적용합니다.

| 실험 | 바꾸는 것 | 사용 기법 (강의안 트랙) |
| --- | --- | --- |
| ① 조 제목 포함 | 임베딩할 글 앞에 장·조 제목을 붙임 | 청킹 전략 변경 (트랙 2의 `full_texts` 아이디어, Step 2에서 예고한 약점 대응) |
| ② E5로 교체 | 임베딩 모델 BGE-M3 → multilingual-e5-base | 임베딩 모델 교체 (트랙 3 : prefix·512토큰·정규화 규칙 준수) |
| ③ 리랭커 도입 | 1단계 후보 20개를 크로스 인코더로 재정렬 | 리랭커 도입 (트랙 5) |
| ④ ⑤ ⑥ 후보 개수 | 리랭커에 넘기는 후보를 5 / 10 / 전체(25)로 조정 | 후보 개수 조정 (트랙 5) |
| ⑦ 조합 | ① + ③ | 두 기법의 조합 |
| top-k 조정 | Hit@1 / Hit@3 / Hit@5 열을 비교 | top-k 조정 |

**실험 조건은 모두 같은 18개 질문**으로 측정합니다. 이 표의 숫자가 곧 결과 해석의 근거입니다.

#### 준비 1 : 리랭커 로드

리랭커도 로컬 오픈소스 모델(`BAAI/bge-reranker-v2-m3`)을 씁니다. 약 2.2GB이므로 VRAM을 먼저 확인합니다.

In [ ]:
# VRAM 확인. 리랭커는 약 2.2GB가 필요합니다
print(f"현재 VRAM : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# 여유가 부족하면 아래 두 줄로 LLM을 내렸다가 나중에 다시 올리세요
# del model
# torch.cuda.empty_cache()

현재 VRAM : 9.45 GB


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

# 동작 확인 : 같은 질문에 대한 두 문서의 점수를 비교합니다
pairs = [
    ("재계약 신청은 언제까지 해야 하나요?", chunk_texts[id_to_idx["제12조"]]),   # 정답 조항
    ("재계약 신청은 언제까지 해야 하나요?", chunk_texts[id_to_idx["제1조"]]),    # 무관한 조항
]
print(reranker.predict(pairs))

# [변경] 첫 실행에는 GPU 준비 시간이 섞이므로, 시간을 재기 전에 한 번 미리 돌려 둡니다
_ = reranker.predict([("워밍업", "워밍업")], show_progress_bar=False)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

[6.1248910e-01 1.6370053e-05]


#### 준비 2 : 실험 도구

새로 만든 코드는 이 부분뿐입니다. 임베딩 모델 두 개를 같은 방식으로 다루는 `embedders`, 실험용 컬렉션을 만드는 `build_collection`, 조건을 받아 18개 질문을 채점하는 `run_experiment`입니다. 안의 검색 코드는 강의안의 `two_stage_retrieve`와 같은 구조입니다.

In [ ]:
# 임베딩 모델을 "같은 방식"으로 부르기 위한 포장
# E5는 문서에 "passage: ", 질문에 "query: "를 붙이는 규칙이 있어서 여기서 한 번만 처리합니다 (트랙 3의 실패 2 예방)
embedders = {
    "bge": {
        "dim": 1024,
        "docs": lambda texts: np.array(bge_embeddings.embed_documents(texts)),
        "query": lambda q: bge_embeddings.embed_query(q),
    },
    "e5": {
        "dim": 768,
        "docs": lambda texts: np.array(e5_embeddings.embed_documents([f"passage: {t}" for t in texts])),
        "query": lambda q: e5_embeddings.embed_query(f"query: {q}"),
    },
}


def build_collection(name, texts, embedder_key):
    """texts 를 임베딩해서 새 컬렉션에 저장합니다 (Step 4 의 컬렉션 생성 + upsert 를 함수로 묶은 것)"""
    vecs = embedders[embedder_key]["docs"](texts)
    if client.collection_exists(name):
        client.delete_collection(name)
    client.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=embedders[embedder_key]["dim"], distance=Distance.COSINE),
    )
    client.upsert(
        collection_name=name,
        points=[
            PointStruct(
                id=i,
                vector=vecs[i].tolist(),
                payload={"text": texts[i], "조항": chunks[i].metadata["조항"], "장": chunks[i].metadata["장"], "id_label": chunk_ids[i]},
            )
            for i in range(len(texts))
        ],
    )
    print(f"컬렉션 '{name}' : {len(texts)}개 저장 (모델 {embedder_key}, {embedders[embedder_key]['dim']}차원)")


question_types = [t for _, _, t in eval_set]      # 질문별 유형 목록 ("의미"/"식별자")


def hit_rate(ranks, k, qtype=None):
    """정답 순위(ranks)가 k 이하인 질문의 비율. qtype 을 주면 그 유형만 계산합니다"""
    idx = [i for i, t in enumerate(question_types) if qtype is None or t == qtype]
    return sum(1 for i in idx if ranks[i] is not None and ranks[i] <= k) / len(idx)


def run_experiment(collection, embedder_key, use_reranker=False, candidate_k=20):
    """18개 질문을 검색해서 질문별 정답 순위와 평균 시간을 돌려줍니다"""
    n_all = len(chunks)
    ranks = []                                   # 질문별 정답 조항의 순위 (후보 밖이면 None)
    start = time.perf_counter()
    for q, gold_label, qtype in eval_set:
        q_vec = embedders[embedder_key]["query"](q)
        # 리랭커를 쓰면 1단계에서 candidate_k개만, 안 쓰면 전체 순위를 보려고 전부 가져옵니다
        limit = candidate_k if use_reranker else n_all
        hits = client.query_points(collection_name=collection, query=q_vec, limit=limit).points

        if use_reranker:
            # 2단계 : (질문, 후보 본문) 쌍을 크로스 인코더로 재채점해서 점수 큰 순서로 줄 세웁니다
            pairs = [(q, h.payload["text"]) for h in hits]
            scores = reranker.predict(pairs, show_progress_bar=False)
            hits = [h for _, h in sorted(zip(scores, hits), key=lambda x: x[0], reverse=True)]

        labels = [h.payload["id_label"] for h in hits]                           # 순위 순서대로 조 이름표
        ranks.append(labels.index(gold_label) + 1 if gold_label in labels else None)
    ms = (time.perf_counter() - start) / len(eval_set) * 1000
    return ranks, ms

print("실험 도구 준비 완료")

실험 도구 준비 완료


#### 준비 3 : 실험용 컬렉션 만들기

- **제목 포함 컬렉션** : `f"{장} {조항}\n{본문}"`. 조 번호가 본문에 없어서 식별자 질문이 약하다는 Step 2의 예고를 확인하려는 것입니다. (강의안 트랙 2의 `full_texts`와 같은 코드입니다.)
- **E5 컬렉션** : 768차원이므로 별도 컬렉션이 필요합니다. "임베딩 모델과 컬렉션은 한 몸"입니다.

In [ ]:
# 강의안 트랙 2의 full_texts : 조 번호(제목)를 본문 앞에 붙인 글
full_texts = [
    f"{c.metadata.get('장','')} {c.metadata.get('조항','')}\n{c.page_content}"
    for c in chunks
]
print(full_texts[id_to_idx["제12조"]][:120])

build_collection("exp_bge_title", full_texts, "bge")     # ① 조 제목 포함 (BGE-M3)
build_collection("exp_e5_body", chunk_texts, "e5")       # ② E5 교체 (본문만)

제3장 입학사정관의 임용 및 위촉 제12조(재계약)
① 전임입학사정관은 제2조 1항의 입학사정관제 지원 사업을 계속하는 경우, 그 업무 추진실적과 근무성적 평가결과에 따라 재계약을 통하여 계약기간을 연장할 수 있다.
컬렉션 'exp_bge_title' : 25개 저장 (모델 bge, 1024차원)
컬렉션 'exp_e5_body' : 25개 저장 (모델 e5, 768차원)


#### 실험 실행

기준선(`COLLECTION`)을 포함해 8개 조건을 한 번에 돌리고 표로 정리합니다. 시간(ms/질문)은 임베딩·검색·리랭킹을 모두 포함한 참고 값입니다.

In [ ]:
# (실험 이름, 변경 내용, 컬렉션, 임베딩 모델, 리랭커 사용, 후보 개수)
EXPERIMENTS = [
    ("기준선",           "조 본문 + BGE-M3, 밀집 검색만",                         COLLECTION,      "bge", False, None),
    ("① 조 제목 포함",   "임베딩할 글 앞에 장·조 제목을 붙임",                      "exp_bge_title", "bge", False, None),
    ("② E5로 교체",      "임베딩 모델 BGE-M3 → multilingual-e5-base (prefix 준수)", "exp_e5_body",   "e5",  False, None),
    ("③ 리랭커(후보 20)", "1단계 후보 20개 → bge-reranker-v2-m3 재정렬",            COLLECTION,      "bge", True,  20),
    ("④ 후보 5",         "리랭커 후보 개수를 20 → 5로 줄임",                        COLLECTION,      "bge", True,  5),
    ("⑤ 후보 10",        "리랭커 후보 개수를 20 → 10으로 조정",                     COLLECTION,      "bge", True,  10),
    ("⑥ 후보 전체(25)",  "리랭커 후보 개수를 20 → 25(전체 청크)로 늘림",             COLLECTION,      "bge", True,  len(chunks)),
    ("⑦ 제목 포함 + 리랭커", "① 조 제목 포함 + ③ 리랭커(후보 20)",                  "exp_bge_title", "bge", True,  20),
]

results = []       # 표의 한 줄씩
all_ranks = {}     # 실험별 질문별 정답 순위 (아래 질문별 표에 사용)

for name, change, collection, emb_key, use_rr, cand_k in EXPERIMENTS:
    ranks, ms = run_experiment(collection, emb_key, use_reranker=use_rr, candidate_k=cand_k or 20)
    all_ranks[name] = ranks
    results.append({
        "실험": name, "변경 내용": change,
        "Hit@1": round(hit_rate(ranks, 1) * 100, 1),
        "Hit@3": round(hit_rate(ranks, 3) * 100, 1),
        "Hit@5": round(hit_rate(ranks, 5) * 100, 1),
        "Hit@3 의미": round(hit_rate(ranks, 3, "의미") * 100, 1),
        "Hit@3 식별자": round(hit_rate(ranks, 3, "식별자") * 100, 1),
        "ms/질문": round(ms),
    })

results_df = pd.DataFrame(results)
base_hit3 = results_df.loc[0, "Hit@3"]
results_df["Hit@3 기준선 대비(%p)"] = (results_df["Hit@3"] - base_hit3).round(1)     # 기준선과의 차이
display(results_df)

# 검산 : Step 6 에서 잰 기준선과 같은 값이어야 합니다
print(f"\nStep 6 기준선 Hit@3 : {baseline_hit3 * 100:.1f}%  /  Step 7 기준선 Hit@3 : {base_hit3}%")

,실험,변경 내용,Hit@1,Hit@3,Hit@5,Hit@3 의미,Hit@3 식별자,ms/질문,Hit@3 기준선 대비(%p)
0,기준선,"조 본문 + BGE-M3, 밀집 검색만",77.8,88.9,88.9,100.0,66.7,24,0.0
1,① 조 제목 포함,임베딩할 글 앞에 장·조 제목을 붙임,83.3,83.3,83.3,100.0,50.0,25,-5.6
2,② E5로 교체,임베딩 모델 BGE-M3 → multilingual-e5-base (prefix 준수),66.7,77.8,88.9,91.7,50.0,14,-11.1
3,③ 리랭커(후보 20),1단계 후보 20개 → bge-reranker-v2-m3 재정렬,88.9,88.9,94.4,100.0,66.7,1093,0.0
4,④ 후보 5,리랭커 후보 개수를 20 → 5로 줄임,83.3,88.9,88.9,100.0,66.7,312,0.0
5,⑤ 후보 10,리랭커 후보 개수를 20 → 10으로 조정,83.3,88.9,88.9,100.0,66.7,584,0.0
6,⑥ 후보 전체(25),리랭커 후보 개수를 20 → 25(전체 청크)로 늘림,88.9,88.9,94.4,100.0,66.7,1254,0.0
7,⑦ 제목 포함 + 리랭커,① 조 제목 포함 + ③ 리랭커(후보 20),100.0,100.0,100.0,100.0,100.0,1037,11.1



Step 6 기준선 Hit@3 : 88.9%  /  Step 7 기준선 Hit@3 : 88.9%


표를 읽는 법입니다. 단위는 %이고, Hit@3 열이 루브릭의 기준값입니다. "Hit@3 기준선 대비"가 양수면 좋아진 것, 0이면 효과 없음, 음수면 나빠진 것입니다. Hit@3이 같아도 **Hit@1**이나 **식별자/의미 열**에서 차이가 날 수 있으니 함께 보세요.

In [ ]:
# 질문별 정답 순위표 : 실험마다 정답 조항이 몇 위였는지 나란히 봅니다 ("후보 밖" = 리랭커에 넘긴 후보 안에 정답이 없었음)
rank_df = pd.DataFrame(
    all_ranks,
    index=[f"[{t}] {q}" for q, _, t in eval_set],
).astype("Int64")
rank_df.insert(0, "정답", [g for _, g, _ in eval_set])
display(rank_df.astype(object).where(rank_df.notna(), "후보 밖"))

,정답,기준선,① 조 제목 포함,② E5로 교체,③ 리랭커(후보 20),④ 후보 5,⑤ 후보 10,⑥ 후보 전체(25),⑦ 제목 포함 + 리랭커
[의미] 입학사정관은 어떤 종류가 있어?,제2조,1,1,4,1,1,1,1,1
[의미] 전임 입학사정관 계약은 몇 년이야?,제11조,1,1,1,1,1,1,1,1
[의미] 재계약하려면 언제까지 신청해야 해?,제12조,1,1,1,1,1,1,1,1
[의미] 인사위원회는 몇 명으로 구성돼?,제15조,1,1,2,1,1,1,1,1
[의미] 위원회에서 안건이 통과되려면 몇 명이 찬성해야 해?,제17조,1,1,1,1,1,1,1,1
[의미] 입학사정관을 그만둔 뒤에 학원에서 일할 수 있어?,제21조,1,1,1,1,1,1,1,1
[의미] 입학사정관이 되려면 학위가 필요해?,제9조,1,1,1,1,1,1,1,1
[의미] 초빙입학사정관을 위촉할 때 내야 하는 서류는?,제10조,1,1,1,1,1,1,1,1
[의미] 위원장이 자리를 비우면 누가 대신해?,제16조,1,1,1,1,1,1,1,1
[의미] 소위원회 위원장은 누구야?,제18조,1,1,1,1,1,1,1,1


In [ ]:
# 기준선에서 정답이 3위 밖이었던 질문만 골라, 대신 어떤 조항이 올라왔는지 확인합니다
def top_labels(query, collection, embedder_key, k=3):
    hits = client.query_points(
        collection_name=collection, query=embedders[embedder_key]["query"](query), limit=k
    ).points
    return [h.payload["id_label"] for h in hits]

for i, (q, gold_label, qtype) in enumerate(eval_set):
    base_rank = all_ranks["기준선"][i]
    if base_rank is None or base_rank > 3:
        print(f"({qtype}) {q}")
        print(f"   정답 {gold_label} 의 기준선 순위 : {base_rank}위 → 대신 올라온 조항 : {top_labels(q, COLLECTION, 'bge')}")

(식별자) 제19조 알려줘
   정답 제19조 의 기준선 순위 : 23위 → 대신 올라온 조항 : ['제20조', '제18조', '제17조']
(식별자) 별지4는 어떤 서식이야?
   정답 제12조 의 기준선 순위 : 18위 → 대신 올라온 조항 : ['제20조', '제5조', '제2조']


## Step 8 : 개선된 파이프라인으로 답변 생성

Step 7에서 가장 좋았던 조건으로 파이프라인을 다시 구성해서 답변을 만들어 봅니다. 아래 `FINAL` 설정은 **기본값으로 ⑦(제목 포함 + 리랭커)** 를 넣어 두었고, 표를 보고 더 나은 조건이 있으면 값을 바꿔서 실행하세요.

In [ ]:
# 표(results_df)를 보고 가장 좋았던 조건에 맞게 바꾸세요
FINAL = {"collection": "exp_bge_title", "embedder": "bge", "rerank": True, "candidate_k": 20}


def retrieve_final(query: str, top_k: int = 3) -> list:
    """Step 7 의 조건(FINAL)대로 검색합니다. 돌려주는 모양은 Step 6 의 retrieve 와 같습니다"""
    q_vec = embedders[FINAL["embedder"]]["query"](query)
    limit = FINAL["candidate_k"] if FINAL["rerank"] else top_k
    hits = client.query_points(collection_name=FINAL["collection"], query=q_vec, limit=limit).points
    if FINAL["rerank"]:
        scores = reranker.predict([(query, h.payload["text"]) for h in hits], show_progress_bar=False)
        hits = [h for _, h in sorted(zip(scores, hits), key=lambda x: x[0], reverse=True)]
    return [
        {"조항": h.payload["조항"], "내용": h.payload["text"], "id_label": h.payload["id_label"]}
        for h in hits[:top_k]
    ]


def rag_answer_final(query: str, top_k: int = 3) -> str:
    contexts = retrieve_final(query, top_k)
    context_text = "\n\n".join(f"[{c['조항']}]\n{c['내용']}" for c in contexts)
    user_prompt = f"""다음은 규정에서 검색된 내용입니다.

{context_text}

질문 : {query}"""
    return generate(SYSTEM_PROMPT, user_prompt)


for q in ["제17조가 무슨 내용이야?", "별지4는 어떤 서식이야?", "전임입학사정관 계약은 몇 년이야?"]:
    print(f"Q : {q}")
    print(f"검색된 조항 : {[c['id_label'] for c in retrieve_final(q)]}")
    print(f"A : {rag_answer_final(q)}")
    print("=" * 60)

Q : 제17조가 무슨 내용이야?
검색된 조항 : ['제17조', '제18조', '제16조']
A : 제17조는 인사위원회의 소집 및 운영에 대한 규정입니다. 이 조항에서는 다음과 같은 내용이 포함되어 있습니다:

① 인사위원회는 제14조 2항의 심의․의결사항이 있을 경우, 위원장이 소집합니다.
② 인사위원회는 3분의 2 이상의 출석과 출석위원 과반수의 찬성으로 의결합니다. 징계에 관한 사항은 재적위원 3분의 2 이상의 찬성으로 의결합니다.
③ 위원장은 표결권을 가지며, 가부동수일 때에는 결정권을 가집니다.
④ 필요하다고 판단될 경우, 위원장은 관련 당사자를 출석시켜 진술 또는 의견을 개진하도록 할 수 있습니다.
⑤ 특정위원과 이해관계가 있는 사항에 대한 의결 시, 해당 위원은 그 사항의 의결에 참여하지 않습니다.
⑥ 모든 사항은 회의록에 기록되고, 위원장의 서명 또는 날인을 받아야 합니다.
⑦ 회의 공개 여부, 운영 방법 등 기타 필요한 사항은 별도로 정할 수 있습니다.

이러한 내용들이 제17조에 포함되어 있습니다. (제17조 참조)
Q : 별지4는 어떤 서식이야?
검색된 조항 : ['제12조', '제10조', '제21조']
A : 별지4는 '재계약 신청서'의 서식입니다. 이는 전임입학사정관이 계약기간을 연장하고자 할 때 제출해야 하는 서류 중 하나입니다.
Q : 전임입학사정관 계약은 몇 년이야?
검색된 조항 : ['제11조', '제12조', '제9조']
A : 전임입학사정관의 계약 기간은 1년이며, 제2조 1항의 입학사정관제 지원 사업을 계속하는 경우 그 기간 내에서 연장할 수 있습니다. (제11조 ①항)


Step 8 출력을 보면, 기준선에서는 검색되지 않았던 "별지4는 어떤 서식이야?"가 이번에는 제12조를 1순위로 가져와서 "재계약 신청서" 서식이라고 올바르게 답했습니다. "제17조가 무슨 내용이야?"도 제17조가 1순위로 검색되어 ①~⑦항 내용을 답했습니다. 이 두 질문은 기준선에서 각각 18위, 2위였던 질문입니다. 단, 이 세 질문 모두 Step 7의 평가 질문과 겹치므로 새 질문에서의 성능을 보여 주는 것은 아닙니다.

## Step 9 : 오프라인 검증

이 프로젝트의 핵심 주장은 "완전 로컬"입니다. 인터넷 연결 없이도 검색·답변이 되는지 확인합니다. 두 겹으로 검증합니다.

1. **Hugging Face 오프라인 설정** : `HF_HUB_OFFLINE=1` 환경변수를 켜고, 이미 import 한 라이브러리 안의 설정값(`huggingface_hub.constants.HF_HUB_OFFLINE`)도 함께 켭니다. 환경변수는 라이브러리를 불러오기 **전에** 정해야 효과가 확실하기 때문에, 이미 불러온 뒤에는 상수도 직접 바꿔야 합니다.
2. **네트워크 차단** : 파이썬의 소켓 연결(`socket.socket.connect`)을 막아서, 인터넷 접속을 시도하면 실패하는 상태를 만든 뒤 임베딩 모델을 캐시에서 다시 불러와 봅니다. 접속이 막혀 있는데도 로드와 답변이 되면 "로컬 파일만으로 동작한다"는 증거가 됩니다.

검사가 끝나면 소켓 차단은 `finally` 에서 반드시 원래대로 되돌립니다. (되돌리지 않으면 이후 셀의 다운로드나 코랩 통신이 막힐 수 있습니다.) 실행 후 출력 4가지를 확인하세요. ① 상수 값이 True, ② 인터넷 접속 시도 실패, ③ 모델 재로드 성공, ④ 답변 생성 성공.

In [ ]:
# 오프라인 검증 : 인터넷을 막은 상태에서 (1) 모델 재로드 (2) 검색 (3) 답변이 되는지 확인합니다
import os                 # 환경변수를 다루는 도구
import socket             # 네트워크 연결(소켓)을 다루는 도구. 이 연결을 막아 인터넷 차단 상태를 만듭니다

# ① Hugging Face 에게 "인터넷에서 받지 말고 이미 받은 파일만 써라"고 알립니다
os.environ["HF_HUB_OFFLINE"] = "1"          # 허깅페이스 허브 오프라인 스위치
os.environ["TRANSFORMERS_OFFLINE"] = "1"    # transformers 라이브러리 오프라인 스위치

import huggingface_hub.constants as hf_constants    # 허브 라이브러리가 실제로 읽는 설정값 모음
hf_constants.HF_HUB_OFFLINE = True                  # 이미 import 한 뒤라 환경변수만으로는 부족할 수 있어서 값을 직접 켭니다
print("HF_HUB_OFFLINE 설정값 :", hf_constants.HF_HUB_OFFLINE)    # True 가 나와야 켜진 것입니다

# ② 인터넷 연결 자체를 막습니다 (검사가 끝나면 아래 finally 에서 원래대로 되돌립니다)
_original_connect = socket.socket.connect           # 원래의 연결 함수를 보관해 둡니다 (나중에 복구하려고)

def _blocked_connect(self, *args, **kwargs):
    raise OSError("[오프라인 검증] 네트워크 연결이 차단되었습니다")    # 접속을 시도하면 오류를 내게 합니다

socket.socket.connect = _blocked_connect            # 연결 함수를 "항상 실패하는 함수"로 바꿔 끼웁니다

try:
    # ③ 정말 인터넷이 막혔는지 확인합니다 : 접속을 시도해서 실패해야 정상입니다
    import requests                                  # 웹 주소에 접속하는 도구
    try:
        requests.get("https://huggingface.co", timeout=3)
        print("인터넷 접속 : 성공 → 차단이 안 된 것입니다 (검증 실패)")
    except Exception as e:
        print("인터넷 접속 시도 → 실패 (정상) :", type(e).__name__)

    # ④ 임베딩 모델을 캐시에서 다시 불러옵니다. GPU 여유를 아끼려고 CPU 로 불러옵니다
    offline_embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},              # 검증용이므로 CPU 로 충분합니다
        encode_kwargs={"normalize_embeddings": True},
    )
    test_vec = offline_embeddings.embed_query("재계약은 언제까지 신청해?")     # 질문을 벡터로 바꿔 봅니다
    print(f"임베딩 모델 재로드 성공 (캐시만 사용) : 벡터 {len(test_vec)}차원")
    del offline_embeddings                            # 검증이 끝났으니 메모리에서 지웁니다

    # ⑤ 이미 올려 둔 Qdrant(로컬 파일) + 리랭커 + 로컬 LLM 으로 답변이 나오는지 확인합니다
    for q in ["재계약은 언제까지 신청해야 해?", "제19조 알려줘"]:
        print(f"Q : {q}")
        print(f"검색된 조항 : {[c['id_label'] for c in retrieve_final(q)]}")
        print(f"A : {rag_answer_final(q)}")
        print("=" * 60)
finally:
    socket.socket.connect = _original_connect        # 어떤 경우에도 소켓 함수를 원래대로 되돌립니다
    print("소켓 차단 해제 완료")

### 결과 해석 및 회고

이 절의 숫자는 위 표와 질문별 순위표에서 그대로 가져왔습니다. 18개 질문이므로 질문 1개가 5.6%p입니다.

| 실험 | Hit@1 | Hit@3 | Hit@5 | 의미 Hit@3 | 식별자 Hit@3 | ms/질문 | Hit@3 기준선 대비 |
|---|---|---|---|---|---|---|---|
| 기준선 | 77.8 | 88.9 | 88.9 | 100.0 | 66.7 | 24 | 0.0 |
| ① 조 제목 포함 | 83.3 | 83.3 | 83.3 | 100.0 | 50.0 | 25 | **-5.6** |
| ② E5로 교체 | 66.7 | 77.8 | 88.9 | 91.7 | 50.0 | 14 | **-11.1** |
| ③ 리랭커(후보 20) | 88.9 | 88.9 | 94.4 | 100.0 | 66.7 | 1093 | 0.0 |
| ④ 후보 5 | 83.3 | 88.9 | 88.9 | 100.0 | 66.7 | 312 | 0.0 |
| ⑤ 후보 10 | 83.3 | 88.9 | 88.9 | 100.0 | 66.7 | 584 | 0.0 |
| ⑥ 후보 전체(25) | 88.9 | 88.9 | 94.4 | 100.0 | 66.7 | 1254 | 0.0 |
| ⑦ 제목 포함 + 리랭커 | 100.0 | 100.0 | 100.0 | 100.0 | 100.0 | 1037 | **+11.1** |

#### 1. 어디가 약했는가 : 기준선의 실패는 식별자 질문 두 개뿐입니다

기준선은 Hit@3 88.9%(16/18)입니다. 의미 질문 12개는 전부 맞혔고(100%), 틀린 두 개가 모두 식별자 질문이었습니다("제19조 알려줘" 정답 순위 23위, "별지4는 어떤 서식이야?" 18위). 이 문서에서 검색을 어렵게 만드는 것은 **문장의 뜻이 아니라 '번호'** 였습니다. 원인은 Step 2에서 조 번호를 `###` 헤더(메타데이터)로 옮기면서 **청크 본문에는 "제19조"라는 글자가 없어졌기 때문**입니다. 밀집 임베딩은 문장의 뜻을 벡터로 바꾸는 방식이라, 본문에 없는 번호 글자와는 애초에 맞출 수 없었고, 제19조 청크가 23위(25개 중 뒤에서 세 번째)까지 밀렸습니다. 제20조가 1위로 올라온 것은 "제19조"와 "제20조"가 숫자만 다른 비슷한 글자열이어서 우연히 가까웠을 가능성이 있습니다(원인 확정은 못 했고, 추정입니다). "별지4"는 제12조 본문 안에 있지만, 청크 전체의 뜻(재계약)에 묻혀서 18위였습니다.

#### 2. 좋아진 시도

- **⑦ 제목 포함 + 리랭커 : Hit@3 88.9 → 100.0 (+11.1%p).** 기준선에서 틀린 두 질문(제19조 → 1위, 별지4 → 1위)이 모두 1위가 되었고, Hit@1도 77.8 → 100.0으로 올랐습니다. 이 결과는 두 조각이 **서로의 약점을 채웠기 때문**으로 읽힙니다. 조 제목을 본문 앞에 붙이면 리랭커(질문과 청크를 함께 읽는 모델)가 "제19조"라는 글자를 실제로 볼 수 있게 되고, 리랭커는 ①에서 생긴 순위 흔들림(아래 3번)을 바로잡습니다. 단독으로는 ①이 -5.6, ③이 0.0이었는데 합치니 +11.1이 된 것이 이 실험에서 가장 큰 관찰입니다. 그 이유를 직접 검증한 것은 아니고, 질문별 순위표에서 나온 해석입니다.
- **③ 리랭커(후보 20) : Hit@3은 그대로(0.0), Hit@1은 77.8 → 88.9(+11.1%p).** 표면 숫자(Hit@3)는 같지만 안에서는 이득과 손실이 맞바뀌었습니다. 이득은 "별지4" 18위 → 1위, "별지1의2 서약서" 3위 → 1위입니다. 손실은 "제17조가 무슨 내용이야?"가 2위 → 4위로 밀려 Hit@3에서 탈락한 것과 "제19조"가 후보 20개 밖으로 나가 버린 것입니다.

#### 3. 효과가 없었거나 나빠진 시도와 그 이유

- **① 조 제목 포함 : 88.9 → 83.3 (-5.6%p, 나빠짐).** 예상과 반대였습니다. 좋아진 것도 있습니다("제17조" 2위 → 1위, "제19조" 23위 → 7위). 그러나 "별지1의2 서약서는 언제 제출해?"가 3위 → 6위로 떨어져 Hit@3에서 탈락했고, "별지4"는 18위 → 19위로 오히려 조금 더 나빠졌습니다. 식별자 Hit@3은 66.7 → 50.0입니다. 제목(장 이름과 조 이름)을 모든 청크 앞에 붙이면 청크들이 서로 더 비슷해지고, 밀집 임베딩은 "제19조" 같은 숫자 글자를 정확히 구분하는 데 약하기 때문으로 보입니다. 제목을 붙여도 "제19조"는 7위에 머물러 3위 안에 들지 못했습니다. 즉 **제목을 붙이는 것만으로는 번호 검색 문제가 해결되지 않았습니다.**
- **② E5로 교체 : 88.9 → 77.8 (-11.1%p, 가장 크게 나빠짐).** 기준선에서 틀리지 않던 의미 질문 "입학사정관은 어떤 종류가 있어?"(1위 → 4위)가 새로 틀렸고, 식별자 질문은 기준선과 비슷하게(제19조 21위, 별지4 16위) 실패했으며 "별지1의2"도 5위로 밀렸습니다. E5(768차원, base 크기)가 BGE-M3(1024차원)보다 이 한국어 규정 문서에서는 덜 정확했다는 것까지는 말할 수 있습니다. 다만 E5도 Hit@5는 88.9로 기준선과 같으므로 "정답을 아예 못 찾는다"기보다 "순위가 살짝 낮다"에 가깝습니다. 참고로 Step 3의 트랙 3 실험에서 E5의 prefix(`query: ` / `passage: `)를 빼도 Hit@3은 78%로 같았습니다. 이 18개 질문에서는 prefix 누락이 점수로 드러나지 않았다는 뜻이며, prefix가 필요 없다는 뜻이 아닙니다(이런 오류가 점수에 안 나타난 채 지나가는 것이 침묵 실패입니다).
- **④⑤⑥ 후보 개수 조정 : Hit@3은 모두 88.9로 기준선과 같음(효과 없음).** 그런데 같은 점수 안에서 실패한 질문이 다릅니다. 후보 5·10개일 때는 "별지4"(기준선 18위)가 후보 밖이라 리랭커가 구제하지 못했습니다. 후보 20·25개일 때는 별지4가 구제되지만, 대신 "제17조"가 2위(후보 5) → 3위(후보 10) → 4위(후보 20·25)로 후보가 늘수록 밀려났습니다. 리랭커는 **1단계(밀집 검색)가 후보에 넣은 것만 다시 정렬할 수 있으므로**, 1단계에서 18위였던 정답은 후보를 18개 이상으로 잡아야 구제됩니다. 반대로 후보가 많아지면 정답이 아닌 조가 정답보다 위로 올라오는 경우가 생깁니다. Hit@3이 같아서 "차이가 없다"고 볼 수 있지만, 실제로는 **이득과 손실이 서로 상쇄된 것**입니다.
- **"제19조" 문제는 ③~⑥ 어느 조건에서도 해결되지 않았습니다.** 후보 20개 이하에서는 후보 밖, 후보 25개(전체)에서도 24위였습니다. 리랭커도 청크 본문만 읽는데 본문에 "제19조"가 없기 때문입니다. 제목을 붙인 ⑦에서 처음 1위가 되었으므로, 이 실패의 원인이 검색 알고리즘이 아니라 **청크에 담긴 글의 내용(번호 글자의 유무)** 이라는 점이 확인됩니다.

#### 4. 후보 개수와 시간의 교환

질문 하나당 시간은 기준선 24ms, 후보 5개 312ms, 10개 584ms, 20개 1093ms, 25개 1254ms입니다. 후보 1개를 추가할 때마다 약 47ms가 늘어나는 정도로 거의 일정했습니다(후보 5 → 25 구간 기준 (1254-312)/20). 리랭커를 켜면 기준선보다 약 40~50배 느려지는 것입니다. 이 문서는 청크가 25개뿐이라 후보를 "전체"로 잡아도 1초대이지만, 문서가 수천 청크로 커지면 후보 수를 그대로 늘릴 수 없고, 그때는 1단계 검색이 정답을 후보 안에 넣어 주는 것이 더 중요해집니다. 또 이 문서에서는 25개 중 20개를 후보로 잡은 것이므로 "후보를 좁혔을 때의 이득"이 사실상 측정되지 않았습니다. 이 한계는 아래 5번과 함께 봐야 합니다.

#### 5. 평가 결과를 믿을 수 있는 범위 (한계)

- 질문이 18개뿐이라 질문 1개가 5.6%p입니다. ⑦의 100%는 "18개 중 18개 정답"이지 실제 사용에서의 성능을 뜻하지 않습니다.
- ⑦은 ①과 ③의 실패 질문을 본 뒤에 만든 조합이고, 같은 18개 질문으로 다시 채점했습니다. **평가 세트에 맞춰 조건을 고른 것**이므로 점수가 실제보다 높게 나왔을 가능성이 큽니다. 새 질문으로 다시 재봐야 하는 이유입니다.
- ⑦에서 "별지4"의 1단계 순위는 19위로, 후보 20개 안에 겨우 들어온 것입니다. 후보를 15개로 줄였다면 실패했을 것이므로 ⑦의 성공은 후보 개수 설정에 민감합니다.
- 정답을 조 하나로 정했지만, 문서에는 한 내용이 여러 조에 걸쳐 나옵니다. 예를 들어 서약서는 제10조(구비서류)에도 나오고, 기준선의 "별지1의2 서약서" 질문 top3에 제10조도 들어왔습니다. 이런 질문은 정답이 여러 개일 수 있는데 지금은 하나만 정답으로 처리하고 있어서 점수가 낮게 나올 수 있습니다.
- 식별자 질문이 6개뿐이어서, 식별자 유형의 점수(66.7 → 100.0)는 개수 몇 개의 변화입니다.

#### 6. 이 문서에서 배운 것

이 규정 문서는 **의미 질문은 기본 설정으로 충분했고, 어려운 것은 번호·서식 이름 같은 식별자 질문**이었습니다. 문서의 성격(번호가 붙은 조항)과 질문의 성격(번호로 묻는 질문)이 결과를 가장 크게 좌우했고, 모델 교체(E5)나 리랭커 후보 개수 조정 같은 "모델 쪽 조정"보다 **청크에 무엇을 담을지(제목 포함 여부)** 가 더 큰 변수였습니다. 다만 제목 포함은 리랭커와 함께 썼을 때만 이득이 되었고 혼자서는 오히려 나빠졌습니다.